# SpanCall Lean Flow — Train / Eval / Test Notebook (No Abstain Machinery)

This notebook runs the **standalone SpanCall Lean workflow** (`spancall/lean/`):
**git clone/pull → env + HF dataset pull → 0-indexed Lean tensorize → T2 training (`LeanSpanCallModel` + `LeanLossContainer`) → `LeanResolver` prediction → offline eval (Part 1 / Part 2 / Part 3) → OOD/adversarial test splits**.

### What Lean Flow Changes vs. Legacy SpanCall
- **Model (`LeanSpanCallModel`)**: Drops `AbstainHead` (H4), `CallAcceptHead`, `CorrectnessHead`, and `abstain_proj` column-0 on the pointer head. Keeps only `backbone` + `ToolSelectionHead` (H1) + `LeanPointerHead` (H3 over `0..C-1` real candidates + `<|null|>`).
- **Loss (`LeanLossContainer`)**: Pure $L = \lambda_{\text{ptr}} L_{\text{ptr}} + \lambda_{\text{tool}} L_{\text{tool}}$ with multi-positive softmax over `0..C-1`.
- **Decoder (`LeanJointDecoder`)**: Always selects the best valid non-overlapping span assignment (with R4 equal-value aggregation + greedy top-1 fallback so overlapping utterance candidates never wipe out parameters).
- **Resolver (`LeanResolver`)**: Direct argmax decoding + type casting (`P9`) + schema validation (`P10`) — no calibration or confidence/abstain gates needed.


---
## 0. Setup — Pull Code from Git & Install

Clones the public SpanCall repository (`https://github.com/eleluong/spancall.git`) or pulls the latest updates from `origin/main` if already cloned (e.g. running in Google Colab), enters the repository directory, adds it to `sys.path`, and installs dependencies in editable mode.

In [9]:
# 0a. Pull latest code from public GitHub repository and install dependencies
import os, sys
from pathlib import Path

GIT_REPO_URL = "https://github.com/eleluong/spancall.git"
GIT_BRANCH   = "main"

# Locate or clone repository
cwd = Path.cwd().resolve()
if (cwd / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd
elif (cwd.parent / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd.parent
elif (cwd / "spancall" / "spancall" / "__init__.py").is_file():
    PROJECT_ROOT = cwd / "spancall"
else:
    print(f"Cloning public repository from {GIT_REPO_URL}...")
    !git clone --branch {GIT_BRANCH} {GIT_REPO_URL}
    PROJECT_ROOT = cwd / "spancall"

# Change directory to project root and ensure it is on sys.path
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Pull latest code from git
if (PROJECT_ROOT / ".git").is_dir():
    print(f"Pulling latest code from {GIT_BRANCH}...")
    !git pull origin {GIT_BRANCH} || echo "git pull note: continuing with local files"

print(f"SpanCall project root: {PROJECT_ROOT}")

# Install package in editable mode along with optional dependencies
%pip install -q -e ".[model,data,agent,dev]" huggingface_hub

Pulling latest code from main...
From https://github.com/eleluong/spancall
 * branch            main       -> FETCH_HEAD
Already up to date.
SpanCall project root: /content/spancall
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for spancall (pyproject.toml) ... done


### 0b. Configuration & Environment

Reads `.env` (or environment variables) for `DATASET_NAME` (`convoicon/spancall`) and `HUGGING_FACE_TOKEN`, auto-discovers `PROJECT_ROOT`, and configures isolated Lean output paths (`data/tensorized_lean` and `results_lean`) so Lean artifacts never overwrite legacy runs.


In [10]:
import os, sys
from pathlib import Path

# Auto-discover project root (works whether cwd is repo root, notebooks/, or /content/spancall)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "spancall" / "__init__.py").is_file():
    if (PROJECT_ROOT.parent / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT / "spancall" / "spancall" / "__init__.py").is_file():
        PROJECT_ROOT = PROJECT_ROOT / "spancall"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spancall.agent.backends.litellm_adapter import _load_dotenv_values

ENV = {**_load_dotenv_values(PROJECT_ROOT / ".env"), **os.environ}
HF_DATASET_ID = ENV.get("DATASET_NAME", "convoicon/spancall")
HF_TOKEN      = ENV.get("HUGGING_FACE_TOKEN") or ENV.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# --- optional Colab Drive persistence ---
DRIVE_MOUNT   = False
DRIVE_OUT     = "/content/drive/MyDrive/spancall_lean_artifacts"

# --- data / checkpoints / eval (isolated from legacy directories) ---
DATA_DIR          = str(PROJECT_ROOT / "data" / "processed")
TENSOR_DIR        = str(PROJECT_ROOT / "data" / "tensorized_lean")
RESULTS_DIR       = str(PROJECT_ROOT / "results_lean")
LEGACY_RESULTS_DIR = str(PROJECT_ROOT / "results")
EVAL_SPLIT        = "hermes_eval"          # in-domain eval (TEST_ID)
TEST_SPLITS       = ["hermes_test_ood_schema", "hermes_test_ood_para", "hermes_test_adversarial"]

import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# ============================================================================
# SINGLE SOURCE OF TRUTH - Lean Flow configuration
# ============================================================================

EXPERIMENT_NAME = "spancall_lean_colab"
SEED            = 42

# --- model / tensorization budget ---
BACKBONE       = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LEN    = 4096
MAX_SPAN_WIDTH = 16
MAX_CANDS      = 512

import yaml
_cfg_yaml = {}
try:
    _cfg_yaml = yaml.safe_load(open("configs/default.yaml")) or {}
except Exception as _cfg_exc:  # noqa: BLE001 - reported, never silent
    print(f"[!] configs/default.yaml unreadable ({type(_cfg_exc).__name__}: "
          f"{_cfg_exc}); max_pos/max_tools/jitter fall back to code defaults.")
MAX_POS = int(((_cfg_yaml.get("optimization") or {}).get("max_pos", 3)) or 3)

# R0.1: ``pack_tools`` is a FLOOR (T = max(pack_tools, 1 + len(native tools))),
# never a cap -- a native tool must never be silently dropped from the prompt.
# ``MAX_TOOLS`` is the one knob allowed to reduce T, and it is an emergency
# brake for VRAM, so it is None (off) by default.
# R1c: ``JITTER_PACK_TOOLS`` draws T uniformly from [1, floor] per row, which is
# what puts the oracle-tool eval layout (T=1) back in the training distribution.
_data_cfg = _cfg_yaml.get("data") or {}
MAX_TOOLS = _data_cfg.get("max_tools")
MAX_TOOLS = None if MAX_TOOLS in (None, "", 0) else int(MAX_TOOLS)
JITTER_PACK_TOOLS = bool(_data_cfg.get("jitter_pack_tools", True))

# Lean Flow only needs multi-tool `t2` (train) and `dev` (held-out eval)
SPLIT_INPUTS = {
    "t2":  "hermes_train.jsonl",
    "dev": "hermes_dev.jsonl",
}

SAMPLE_LIMITS = {"t2": None, "dev": None}
DATA_FRACTION = 0.2
TENSORIZE_WORKERS = max(1, min(os.cpu_count() or 1, 16))

EVAL_LIMIT     = 500
TEST_LIMIT     = 200
FORCE_HF_DOWNLOAD = False

print("=" * 72)
print("LEAN FLOW DATA CONFIG")
print("=" * 72)
print(f"EXPERIMENT_NAME   = {EXPERIMENT_NAME}  SEED = {SEED}")
print(f"BACKBONE          = {BACKBONE}")
print(f"MAX_SEQ_LEN       = {MAX_SEQ_LEN}  MAX_SPAN_WIDTH = {MAX_SPAN_WIDTH}  MAX_POS = {MAX_POS}")
print(f"MAX_TOOLS         = {MAX_TOOLS}  (None = no ceiling; pack_tools is a floor, never a cap)")
print(f"JITTER_PACK_TOOLS = {JITTER_PACK_TOOLS}  (per-row T draw from [1, floor] so T=1 is in-distribution)")
for k, v in SPLIT_INPUTS.items():
    print(f"  {k:<9} <- {v}   (limit={SAMPLE_LIMITS.get(k)})")
print(f"DATA_FRACTION     = {DATA_FRACTION}")
print(f"TENSORIZE_WORKERS = {TENSORIZE_WORKERS}")
print(f"EVAL_LIMIT        = {EVAL_LIMIT}  TEST_LIMIT = {TEST_LIMIT}")
print(f"DATA_DIR          = {DATA_DIR}")
print(f"TENSOR_DIR        = {TENSOR_DIR}")
print(f"RESULTS_DIR       = {RESULTS_DIR}")
print(f"DEVICE            = {DEVICE}")


LEAN FLOW DATA CONFIG
EXPERIMENT_NAME   = spancall_lean_colab  SEED = 42
BACKBONE          = Qwen/Qwen3-Embedding-0.6B
MAX_SEQ_LEN       = 4096  MAX_SPAN_WIDTH = 16  MAX_POS = 1
MAX_TOOLS         = None  (None = no ceiling; pack_tools is a floor, never a cap)
JITTER_PACK_TOOLS = True  (per-row T draw from [1, floor] so T=1 is in-distribution)
  t2        <- hermes_train.jsonl   (limit=None)
  dev       <- hermes_dev.jsonl   (limit=None)
DATA_FRACTION     = 0.2
TENSORIZE_WORKERS = 12
EVAL_LIMIT        = 500  TEST_LIMIT = 200
DATA_DIR          = /content/spancall/data/processed
TENSOR_DIR        = /content/spancall/data/tensorized_lean
RESULTS_DIR       = /content/spancall/results_lean
DEVICE            = cuda


---
## 1. Runtime & Package Check

Verify GPU/device availability and confirm `spancall.lean` and `span_template` import cleanly.

In [11]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'No NVIDIA GPU detected (using MPS / CPU fallback)'

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [12]:
import spancall
import spancall.lean as lean
import span_template
import transformers
import datasets
import huggingface_hub
from spancall.lean.eval_p1 import (
    EXTRACTABLE_TARGET_TYPES,
    NON_EXTRACTABLE_TARGET_TYPES,
    e3_aligner_violation,
    eval_param_extraction_lean,
)

print("spancall     :", spancall.__file__)
print("spancall.lean:", lean.__file__)
print("span_template:", span_template.__file__)
print("transformers :", transformers.__version__, "| datasets:", datasets.__version__)
print("retrieval-first eval ready; extractable target types:", sorted(EXTRACTABLE_TARGET_TYPES))
print("out-of-scope target types:", sorted(NON_EXTRACTABLE_TARGET_TYPES),
      "| E3 tripwire:", callable(e3_aligner_violation))

spancall     : /content/spancall/spancall/__init__.py
spancall.lean: /content/spancall/spancall/lean/__init__.py
span_template: /content/spancall/span_template/__init__.py
transformers : 5.22.1 | datasets: 4.8.5
retrieval-first eval ready; extractable target types: ['boolean_ctx', 'enum_ctx', 'null_ctx', 'recent_ctx', 'time_ctx', 'utterance_span']
out-of-scope target types: ['unresolved'] | E3 tripwire: True


### 1b. (Optional) Google Drive mount (Colab only)

Mounted artifacts under `DRIVE_OUT` keep results between Colab sessions.

In [13]:
if DRIVE_MOUNT:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('drive mounted:', DRIVE_OUT)

---
## 2. Dataset — Load via Hugging Face (`convoicon/spancall`)

Loads the 7 canonical SpanCall splits (`hermes_train`, `hermes_dev`, `hermes_calib`, `hermes_eval`, `hermes_test_ood_schema`, `hermes_test_ood_para`, `hermes_test_adversarial`) from `HF_DATASET_ID` (`convoicon/spancall`) on the Hugging Face Hub.

In [14]:
from huggingface_hub import snapshot_download
from datasets import load_dataset

CANONICAL_SPLITS = [
    "hermes_train.jsonl",
    "hermes_dev.jsonl",
    "hermes_calib.jsonl",
    "hermes_eval.jsonl",
    "hermes_test_ood_schema.jsonl",
    "hermes_test_ood_para.jsonl",
    "hermes_test_adversarial.jsonl",
]

os.makedirs(DATA_DIR, exist_ok=True)
missing = [f for f in CANONICAL_SPLITS if not os.path.exists(os.path.join(DATA_DIR, f))]

if FORCE_HF_DOWNLOAD or missing:
    print(f"Downloading splits from Hugging Face dataset: {HF_DATASET_ID} ...")
    snapshot_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        local_dir=str(PROJECT_ROOT),
        allow_patterns=["data/processed/*.jsonl", "data/processed/*.json"],
        token=HF_TOKEN,
    )

print("Canonical splits in DATA_DIR:")
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:<34} ({os.path.getsize(p):>12,} bytes)")

hf_ds = load_dataset(
    "json",
    data_files={
        "train": os.path.join(DATA_DIR, "hermes_train.jsonl"),
        "dev":   os.path.join(DATA_DIR, "hermes_dev.jsonl"),
        "eval":  os.path.join(DATA_DIR, "hermes_eval.jsonl"),
    },
)
print("\nHugging Face DatasetDict loaded:", hf_ds)

Canonical splits in DATA_DIR:
  hermes_calib.jsonl                 (   4,644,993 bytes)
  hermes_dev.jsonl                   (   4,624,634 bytes)
  hermes_eval.jsonl                  (  15,616,323 bytes)
  hermes_test_adversarial.jsonl      ( 129,893,828 bytes)
  hermes_test_ood_para.jsonl         (   5,785,351 bytes)
  hermes_test_ood_schema.jsonl       (  12,061,147 bytes)
  hermes_train.jsonl                 ( 102,226,436 bytes)
  prepare_report.json                (       3,543 bytes)

Hugging Face DatasetDict loaded: DatasetDict({
    train: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 24071
    })
    dev: Dataset({
        features: ['sample_id', 'source_dataset', 'utterance', 'tools', 'ground_truth_calls', 'recent_context', 'current_time_iso', 'stratum', 'target_spans', 'split', 'metadata'],
        num_rows: 1720
  

---
## 3. Lean Tensorizer — `UnifiedSample` → 0-Indexed Lean Trainer Rows

Uses [`spancall.lean.tensorize.tensorize_sample_lean`](../spancall/lean/tensorize.py) directly:
- **0-indexed `pos_indices` in `[0, C)`** (no column-0 abstain slot; `-1` is padding).
- Omitted optional/nullable parameters on Stratum-F rows point at a **deterministic** `<|null|>` candidate slot (resolved per param, never a best-effort region search).
- Rows without any positive groundable target (`has_positive_targets(row) == False`, e.g., empty Stratum-A rows) are filtered out cleanly so every training step has active pointer + tool supervision.

### Param supervision audit (retrieval-first contract)

A Stratum-F param with no matched positive receives **no pointer loss term** (the loss zeroes rows where `n_pos == 0`). That silent exclusion is only correct for some reasons, so the tensorizer classifies **every** drop into a 5-way taxonomy:

| drop reason | meaning | severity |
|---|---|---|
| `unresolved_target` | GT target type is `unresolved` — the value must be generated/transformed, not retrieved | **expected**: out of scope for the fast path |
| `absent_from_gt` | no aligned target and no GT arg — the param is not part of this call | **benign** |
| `match_missed` | an *extractable* target existed but the span matcher failed to find it | **bug**: a groundable param is silently unsupervised — must be driven to ~0 |
| `unaligned_target` | a target row exists but carries neither `canonical_value` nor `char_start` | **bug**: aligner produced an unusable target |
| `null_slot_missing` | GT is null / `null_ctx` but the mask builder emitted no `<|null|>` candidate | **bug**: cannot supervise a known-null answer |

Explicit `null` GT arguments are supervised: they map to the deterministic `<|null|>` slot. The cell below prints every reason per split, flags the two defect classes, and ends with three tripwires — `match_missed total`, `aligner/null-slot defects total`, and `tensorize exceptions total` — all of which must be `0`. Exceptions raised inside the fork worker are now **counted by class** instead of being silently swallowed into a smaller training set.


In [15]:
import json, os, random
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.data.schema import UnifiedSample
from spancall.model.masks import CandidateMaskBuilder
from spancall.lean.tensorize import ToolPool, tensorize_sample_lean, has_positive_targets

tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)
tokenizer.add_special_tokens({"additional_special_tokens": list(SPECIAL_MARKER_TOKENS)})
mask_builder = CandidateMaskBuilder(max_seq_len=MAX_SEQ_LEN, max_span_width=MAX_SPAN_WIDTH)

# ---- H1 candidate breadth -------------------------------------------------
# Hermes rows ship ONE tool each, so packing only s.tools renders a single
# <|tool|> anchor: tool_logits is (B, 1), argmax is always 0, tool_targets is
# always 0 => tool_acc = 1.0 for ANY weights and CE over one class = 0 (H1 gets
# no gradient). PACK_TOOLS > 1 packs real distractor schemas from the corpus so
# H1 becomes an actual discrimination task.
import yaml
# Guarded read: a missing/unparsable config must not kill the notebook, and a
# silent fallback of 1 would reintroduce the vacuous single-candidate H1.
PACK_TOOLS = 1
try:
    with open(os.path.join(str(PROJECT_ROOT), "configs", "default.yaml")) as _fh:
        _default_cfg = yaml.safe_load(_fh) or {}
    PACK_TOOLS = int((_default_cfg.get("data") or {}).get("pack_tools", 1) or 1)
except Exception as _exc:  # noqa: BLE001 - reported, never silent
    print(f"[!] configs/default.yaml unreadable ({type(_exc).__name__}: {_exc}); "
          f"defaulting PACK_TOOLS=1 -> H1 will be a single-class softmax (CE == 0).")
TOOL_POOL = None  # bound (once) in the tensorize cell, before forking
print("tokenizer ready; vocab =", len(tokenizer), "| MAX_SEQ_LEN =", MAX_SEQ_LEN,
      "| PACK_TOOLS =", PACK_TOOLS)

tokenizer ready; vocab = 151684 | MAX_SEQ_LEN = 4096 | PACK_TOOLS = 3


In [16]:
# ---- tensorize splits into Lean 0-indexed format (parallel fork pool) ----
# ``typing`` is imported HERE, in the cell that uses it. Notebook cells do not
# share scope, so the ``from typing import ...`` that appears in the resolver
# cell further down is invisible to this one: annotating ``cand_hist: Dict[...]``
# without this line raises NameError at runtime.
from typing import Any, Dict, List, Optional
import json
import os
import queue
import random
import time

def _iter_raw_lines(path: str):
    """Yield the raw JSONL TEXT of each non-empty line.

    Deliberately does NOT ``json.loads`` here: parsing in the parent would make
    the whole corpus parse single-threaded before any fork happens (the parent
    wasted one full core on this on Colab's 2 vCPUs) and would hold every row
    dict in RAM at once. The fork workers parse their own line instead.
    """
    with open(path, "r") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield line

def _iter_raw_dicts(path: str):
    """Parsed variant, for consumers that need dicts in the parent."""
    for line in _iter_raw_lines(path):
        yield json.loads(line)

def _fraction_subset(raw_lines, frac: float, seed: int):
    """Deterministic fraction of the raw lines (still unparsed text)."""
    raw_lines = list(raw_lines)
    if frac >= 1.0:
        return raw_lines
    rng = random.Random(int(seed))
    n = max(int(len(raw_lines) * frac), 1)
    keep = set(rng.sample(range(len(raw_lines)), min(n, len(raw_lines))))
    return [d for i, d in enumerate(raw_lines) if i in keep]

def _serialize(row):
    """JSON-serializable view of a tensorized row.

    CONTRACT: the worker returns ``json.dumps(_serialize(row))`` and the parent
    writes it verbatim. ``spancall/lean/trainer.py:573 load_rows`` re-parses
    these files, so the bytes must stay identical to the old parent-side
    ``json.dumps(r)``. Do not add kwargs here.
    """
    return {k: (v.tolist() if hasattr(v, "tolist") else v) for k, v in row.items()}

def _tensorize_lean_task(args):
    """Tensorize one sample; returns (json_line_or_None, drop_reasons, errors).

    ``args`` is ``(raw_jsonl_text, pack_multi)``, NOT a parsed dict: the child
    does its own ``json.loads``, so the parent never pays a single-threaded
    corpus parse and never holds the whole split in RAM. Returning an already
    serialized line (rather than a dict) likewise moves ``json.dumps`` off the
    parent, which was the only serial tail left after the pool finished.

    The serialized form is byte-identical to ``json.dumps(_serialize(row))``
    because ``spancall/lean/trainer.py:573`` re-reads these files; see the
    contract note on ``_serialize`` below.

    The drop-reason audit travels back through the return value because the
    fork pool cannot mutate a parent dict. It is reported even for rows that
    are filtered out (all params dropped => no positives), which is exactly
    the case that used to vanish silently.

    Exceptions are NO LONGER swallowed. The exception class name is counted
    and returned so a tokenizer/schema mismatch shows up as a number instead
    of silently shrinking the training set.
    """
    raw_line, pack_multi = args
    drops = {}
    errors = {}
    try:
        d = json.loads(raw_line)
        s = UnifiedSample.from_dict(d)
        ts_stats = {}
        row = tensorize_sample_lean(
            s,
            tokenizer=tokenizer,
            mask_builder=mask_builder,
            max_seq_len=MAX_SEQ_LEN,
            max_cands=MAX_CANDS,
            max_pos=int(MAX_POS),
            pack_multi_tools=pack_multi,
            stats=ts_stats,
            # Read the pool from the MODULE GLOBAL, not the task tuple. Under the
            # "fork" context every child already inherits the parent's globals,
            # so passing the pool in the tuple only pickled the entire schema
            # corpus once per row (measured: 4.2 GB of redundant pickling on a
            # 50k-row split). Rebind ``TOOL_POOL``/``PACK_TOOLS_EPH`` before
            # calling _tensorize_many; the fallback path reads the same globals.
            tool_pool=TOOL_POOL,
            pack_tools=PACK_TOOLS_EPH,
            # R0.1: an explicit, loud ceiling that can only REDUCE T. ``None``
            # (the default) means "never drop a native tool"; setting it is an
            # emergency VRAM brake, and the truncation is counted below.
            max_tools=MAX_TOOLS,
            # R1c: draw T per row from [1, floor] so the oracle-tool eval layout
            # (T=1) is inside the training distribution instead of out of it.
            jitter=JITTER_PACK_TOOLS,
        )
        # L1 audit: classify Stratum-F params that ended up with NO positive.
        #   unresolved_target -> needs generation/transform; OUT OF SCOPE for the
        #                        retrieval-first fast path (correct exclusion).
        #   unaligned_target  -> target row carried neither value nor span (data bug).
        #   null_slot_missing -> answer is null but no <|null|> candidate exists (bug).
        #   absent_from_gt    -> param is not part of this call (benign).
        #   match_missed      -> groundable target existed but the matcher failed
        #                        => a supervision bug, silently excluded from loss.
        drops = dict(ts_stats.get("param_drop_reasons", {}))
        # Anchor-snap misses must be visible: a bridge_text tool line the
        # tokenizer could not map back to tokens means that H1 candidate was
        # emitted as PAD_INDEX (-1) and is invisible to the tool head.
        # This MUST be merged AFTER the rebind above: writing it into ``drops``
        # before ``drops = dict(ts_stats[...])`` discarded it unconditionally,
        # so the downstream tool_snap_miss warnings could never fire.
        if ts_stats.get("tool_snap_misses"):
            drops["tool_snap_miss"] = drops.get("tool_snap_miss", 0) + int(
                ts_stats["tool_snap_misses"]
            )
        # R9: the render-T histogram must survive the return trip. The fork pool
        # cannot mutate a parent dict, and ``drops`` is the only counter channel
        # that already crosses the process boundary, so encode the histogram as
        # synthetic ``render_T=<T>`` keys and decode it in the parent.
        for _t, _cnt in (ts_stats.get("render_T_hist") or {}).items():
            _k = f"render_T={int(_t)}"
            drops[_k] = drops.get(_k, 0) + int(_cnt)
        # R0.1: a truncated row is a model-visible capability loss, not a benign
        # data quirk -- count it on the same channel so it cannot vanish.
        if ts_stats.get("tool_truncated_rows"):
            drops["tool_truncated_rows"] = drops.get("tool_truncated_rows", 0) + int(
                ts_stats["tool_truncated_rows"]
            )
        # R1c: how many rows actually had their floor jittered.
        if ts_stats.get("jittered_rows"):
            drops["jittered_rows"] = drops.get("jittered_rows", 0) + int(
                ts_stats["jittered_rows"]
            )
        if row is None or not has_positive_targets(row):
            return None, drops, errors
        # Serialize inside the child: the parent then only concatenates text.
        return json.dumps(_serialize(row)), drops, errors
    except Exception as exc:  # noqa: BLE001 - counted, not swallowed
        cls = type(exc).__name__
        errors[cls] = errors.get(cls, 0) + 1
        return None, drops, errors

# In-flight task cap. Bounds both the reorder buffer and parent RAM; large
# enough that a straggler row cannot starve the pool, small enough that we do
# not pickle 50k lines into the queue at once.
_WINDOW = 2048

def _drain_ordered(pool, raw, pack_multi: bool, absorb, done, rows_len, need,
                   workers: int = 1, window: int = _WINDOW):
    """Submit in input order, apply in input order, without head-of-line stalls.

    ``ProcessPoolExecutor.map`` yields strictly in submission order, so ONE slow
    row parks every already-finished result behind it and idles the other
    workers (head-of-line blocking). Here results are queued as they complete
    and applied only when every earlier index has been applied -- identical
    output order, but no worker waits on a straggler.

    The completion queue is fed by futures' done callbacks, so the parent does
    no polling: one blocking ``get`` per row, O(1) each. (An earlier version
    called ``wait`` on the whole in-flight set each iteration, which allocated a
    window-sized set per row and burned real parent CPU.)

    Two separate bounds keep submission small:

    * ``window`` caps outstanding tasks so the reorder buffer stays bounded.
    * the *budget* (``need - rows``) caps them by how many rows we can still
      use. Without it a sampled split (``need=200`` of 50k lines) happily
      submitted and pickled the whole corpus before the early stop was ever
      consulted; a ``need=5`` run would still ship 500 tasks to the pool.
      ``guard`` exists because rows can be DROPPED: we may need to consume
      more tasks than ``need`` remaining to still collect ``need`` rows, and
      the cap self-corrects after every completion, so it never deadlocks.

    The budget is charged against ``submit - nxt`` -- tasks SUBMITTED but not
    yet APPLIED -- not against in-flight tasks alone. A finished result parked
    in the reorder buffer is still a row we may be about to collect, so counting
    only in-flight tasks let one extra task per applied row slip past the cap
    (``need=5`` shipped 22 tasks against a 21-task budget).
    """
    n = len(raw)
    guard = max(2 * int(workers), 16)
    finished_q = queue.SimpleQueue()   # (index, future) in COMPLETION order
    buffer = {}                        # index -> result, ready but not applicable yet
    nxt = 0                            # next index whose result may be applied
    submit = 0                         # next index to submit
    inflight = 0
    while nxt < n and not done():
        budget = max(0, need - rows_len())
        cap = min(window, budget + guard)
        while submit < n and (submit - nxt) < cap and not done():
            fut = pool.submit(_tensorize_lean_task, (raw[submit], pack_multi))
            # Bind the index NOW; a closure over `submit` would see its final value.
            fut.add_done_callback(lambda f, i=submit: finished_q.put((i, f)))
            submit += 1
            inflight += 1
        if inflight == 0:
            break
        idx, fut = finished_q.get()
        inflight -= 1
        buffer[idx] = fut.result()
        while nxt in buffer and not done():
            absorb(buffer.pop(nxt))
            nxt += 1

def _tensorize_many(raw, pack_multi: bool, need, workers: int, executor=None):
    """Tensorize raw JSONL lines; returns (rows, drops, errors, info).

    ``rows`` holds pre-serialized JSON strings in input order.
    ``executor`` lets the caller fork ONE pool for every split; without it a
    throwaway pool is created (and torn down) per split.
    """
    if need is None:
        need = len(raw)
    rows = []
    drops_total = Counter()
    errors_total = Counter()
    info = {"workers": 1, "pooled": False}

    def _absorb(res):
        line, drops, errors = res
        for k, v in (drops or {}).items():
            drops_total[k] += v
        for k, v in (errors or {}).items():
            errors_total[k] += v
        if line is not None and len(rows) < need:
            rows.append(line)

    def _done():
        return len(rows) >= need

    ctx = None
    if executor is None and workers > 1:
        try:
            ctx = mp.get_context("fork")
        except Exception as exc:  # noqa: BLE001
            # Silent degradation to 1 vCPU is the failure mode that makes this
            # cell look "just slow" on Colab. Say it out loud.
            print(f"  [!] fork context unavailable ({type(exc).__name__}: {exc}); "
                  f"tensorizing SINGLE-PROCESS with {workers} requested workers")
            ctx = None
    elif executor is None:
        print(f"  [!] TENSORIZE_WORKERS={workers} -> tensorizing SINGLE-PROCESS")

    if executor is not None or ctx is not None:
        pool = executor
        owned = pool is None
        try:
            if owned:
                pool = ProcessPoolExecutor(max_workers=workers, mp_context=ctx)
            info["workers"] = int(getattr(pool, "_max_workers", workers) or workers)
            info["pooled"] = True
            _drain_ordered(pool, raw, pack_multi, _absorb, _done,
                           rows_len=lambda: len(rows), need=need,
                           workers=info["workers"])
            return rows, drops_total, errors_total, info
        except Exception as exc:  # noqa: BLE001 - reported, not swallowed
            print(f"  [!] process pool failed ({type(exc).__name__}: {exc}); "
                  f"falling back to single-process")
            info["workers"] = 1
            info["pooled"] = False
            rows = []
            drops_total = Counter()
            errors_total = Counter()
        finally:
            if owned and pool is not None:
                pool.shutdown(wait=True)
    for line in raw:
        _absorb(_tensorize_lean_task((line, pack_multi)))
        if _done():
            break
    return rows, drops_total, errors_total, info

groups = defaultdict(list)
for out_name, in_name in SPLIT_INPUTS.items():
    pack_multi = True
    groups[(in_name, pack_multi)].append(out_name)

os.makedirs(TENSOR_DIR, exist_ok=True)
stats = {}
drop_summary = {}
error_summary = {}

# ---- ONE distractor pool, built from TRAIN rows only ----------------------
# Two independent reasons this is a single pool hoisted out of the split loop:
#  1. LEAKAGE: building the pool per split seeded the dev/eval distractor set
#     with dev/eval schemas, i.e. held-out schemas influenced the very
#     candidate list the model is scored on. Train-only removes that channel
#     (a dev row's GT tool still comes from the row itself, never the pool).
#  2. The pool is a plain Python object. Binding it ONCE to the module global
#     below lets the fork pool inherit it by reference: it never enters the
#     task tuple, so it is never re-pickled per row.
POOL_SOURCE = SPLIT_INPUTS.get("t2") or next(iter(SPLIT_INPUTS.values()))
_pool_path = os.path.join(DATA_DIR, POOL_SOURCE)
if PACK_TOOLS > 1:
    if not os.path.exists(_pool_path):
        print(f"[!] tool-pool source missing ({_pool_path}) -> H1 distractors "
              f"disabled; tool_acc will be a single-class artifact.")
        TOOL_POOL = None
    else:
        TOOL_POOL = ToolPool.from_row_dicts(_iter_raw_dicts(_pool_path), seed=SEED)
        print(f"[tool-pool] built once from {POOL_SOURCE}: {len(TOOL_POOL)} "
              f"distinct schemas (pack_tools={PACK_TOOLS})")
        if len(TOOL_POOL) < PACK_TOOLS:
            print(f"[!VACUOUS-H1] pool holds {len(TOOL_POOL)} schemas but "
                  f"pack_tools={PACK_TOOLS}: no row can reach T={PACK_TOOLS}. "
                  f"Lower data.pack_tools or ingest more schemas.")
        if TOOL_POOL.n_duplicates:
            print(f"[tool-pool] {TOOL_POOL.n_duplicates} duplicate schema name(s) "
                  f"collapsed; {len(TOOL_POOL.name_collisions)} of them had a "
                  f"DIFFERING definition (first kept): "
                  f"{TOOL_POOL.name_collisions[:10]}")
else:
    TOOL_POOL = None
    print("[tool-pool] pack_tools=1 -> no distractors (H1 will be vacuous: "
          "CE == 0, tool_acc == 1.0 for any weights).")
PACK_TOOLS_EPH = int(PACK_TOOLS) if TOOL_POOL is not None else 1

# ---- ONE pool for ALL splits ---------------------------------------------
# Forking per split re-warmed the interpreter and re-inherited the tokenizer
# every time; on Colab most of a short split's wall-clock was that startup.
# One pool is forked once (inheriting TOOL_POOL/tokenizer by reference) and
# shared by every (in_name, pack_multi) group below.
_fork_ctx = None
_shared_ex = None
if TENSORIZE_WORKERS > 1:
    try:
        _fork_ctx = mp.get_context("fork")
    except Exception as _exc:  # noqa: BLE001 - surfaced, never silent
        print(f"[!] fork context unavailable ({type(_exc).__name__}: {_exc}) -> "
              f"tensorize will run SINGLE-PROCESS on one core; expect a large "
              f"slowdown versus Colab's {os.cpu_count()} vCPU(s).")
if _fork_ctx is not None:
    _shared_ex = ProcessPoolExecutor(max_workers=TENSORIZE_WORKERS, mp_context=_fork_ctx)
    print(f"[tensorize] one fork pool for all splits: {TENSORIZE_WORKERS} worker(s) "
          f"(cpu_count={os.cpu_count()})")

per_split_timing = {}
try:
    for (in_name, pack_multi), out_names in groups.items():
        src = os.path.join(DATA_DIR, in_name)
        if not os.path.exists(src):
            for on in out_names:
                print(f"SKIP {on}: {src} missing")
            continue
        t_read = time.perf_counter()
        raw = _fraction_subset(_iter_raw_lines(src), DATA_FRACTION, SEED)
        t_read = time.perf_counter() - t_read
        need = None
        for on in out_names:
            lim = SAMPLE_LIMITS.get(on)
            if lim is not None:
                need = lim if need is None else max(need, lim)
        t_pool = time.perf_counter()
        rows, drops, errors, tinfo = _tensorize_many(
            raw, pack_multi, need, TENSORIZE_WORKERS, executor=_shared_ex,
        )
        t_pool = time.perf_counter() - t_pool
        per_split_timing[in_name] = {
            "read_lines_s": round(t_read, 3),
            "tensorize_s": round(t_pool, 3),
            "workers": tinfo["workers"],
            "pooled": tinfo["pooled"],
            "rows_per_s": (round(len(rows) / t_pool, 1) if t_pool > 0 else None),
        }
        print(f"[tensorize] {in_name}: {len(rows)} rows in {t_pool:.1f}s "
              f"({len(rows) / t_pool:.1f} rows/s, {tinfo['workers']} worker(s), "
              f"read {t_read:.2f}s)")
        missed = int(drops.get("match_missed", 0))
        unresolved = int(drops.get("unresolved_target", 0))
        drop_summary[in_name] = dict(drops)
        if errors:
            error_summary[in_name] = dict(errors)
        for on in out_names:
            limit = SAMPLE_LIMITS.get(on)
            out_rows = rows if limit is None else rows[:limit]
            out_path = os.path.join(TENSOR_DIR, f"{on}.jsonl")
            t_write = time.perf_counter()
            with open(out_path, "w") as fh:
                # Rows arrive pre-serialized from the workers: no json.dumps
                # on the parent's single thread.
                fh.write("\n".join(out_rows))
                if out_rows:
                    fh.write("\n")
            per_split_timing[in_name]["write_s"] = round(
                time.perf_counter() - t_write, 3)
            # Stats need dicts; parse the kept rows only (not the whole split).
            stats_rows = [json.loads(x) for x in out_rows]
            # LIVE anchors only (>= 0). A row with zero live anchors carries NO
            # H1 supervision, so counting it as "1 candidate" would dilute the
            # mean toward the vacuity threshold and hide a real problem.
            # n_rows_h1 is the honest denominator: rows H1 can learn from.
            cand_counts = [
                int(r.get("_n_tool_candidates", 0)) for r in stats_rows
                if int(r.get("_n_tool_candidates", 0)) > 0
            ]
            cand_hist: Dict[int, int] = {}
            for c in cand_counts:
                cand_hist[c] = cand_hist.get(c, 0) + 1
            mean_cands = ((sum(cand_counts) / len(cand_counts))
                          if cand_counts else float("nan"))
            n_rows_h1 = len(cand_counts)
            n_rows_no_h1 = len(stats_rows) - n_rows_h1
            # R9: the RENDER count T, beside mean_n_candidates. These measure
            # different things and both matter: T is how many <|tool|> blocks the
            # prompt actually carries (the train/serve layout axis), while
            # n_candidates is how many of those anchors survived tokenization.
            # ``tool_positions`` is serialized as a plain list, so len() is the
            # exact T the trainer will see for this row.
            render_hist: Dict[int, int] = {}
            for r in stats_rows:
                tp = r.get("tool_positions")
                if not tp:
                    continue
                render_hist[len(tp)] = render_hist.get(len(tp), 0) + 1
            n_render_rows = sum(render_hist.values())
            mean_render_T = ((sum(t * c for t, c in render_hist.items())
                              / n_render_rows) if n_render_rows else float("nan"))
            t1_share = (render_hist.get(1, 0) / n_render_rows) if n_render_rows else float("nan")
            n_truncated = int(drops.get("tool_truncated_rows", 0))
            n_jittered = int(drops.get("jittered_rows", 0))
            # Phase I: capacity/layout channels. These are COUNTERS, not drop
            # reasons — lumping them into ``param_drop_other`` would print
            # "[note] dropped_over_length=412" as if a parameter had lost its
            # supervision, which is a different (and worse) failure.
            n_over_length = int(drops.get("dropped_over_length", 0))
            n_ambiguous = int(drops.get("multi_call_ambiguous_anchors", 0))
            n_recent_trunc = int(drops.get("recent_truncated_rows", 0))
            n_recent_rows = int(drops.get("n_recent_rows", 0))
            _CHANNEL = (
                "tool_truncated_rows", "jittered_rows", "dropped_over_length",
                "multi_call_ambiguous_anchors", "recent_truncated_rows",
                "n_recent_rows", "tool_snap_misses", "param_snap_misses",
            )
            param_other = {
                k: v for k, v in drops.items()
                if k not in ("match_missed", "unresolved_target")
                and k not in _CHANNEL
                and not k.startswith("render_T=")
            }
            stats[on] = {
                "rows": len(stats_rows),
                "filtered_or_dropped": len(raw) - len(rows),
                "raw_after_fraction": len(raw),
                "fraction": DATA_FRACTION,
                "pack_multi": pack_multi,
                "limit": limit,
                "pack_tools": PACK_TOOLS_EPH,
                "tool_pool_size": len(TOOL_POOL) if TOOL_POOL is not None else 0,
                "rows_with_h1_supervision": n_rows_h1,
                "rows_without_h1_supervision": n_rows_no_h1,
                # NaN (not 0.0) when H1 has nothing to score: 0.0 reads as a
                # real measurement and silently passes any `< 1.5` check.
                "mean_n_candidates": (round(mean_cands, 3)
                                      if mean_cands == mean_cands else None),
                "n_candidates_hist": {str(k): v for k, v in sorted(cand_hist.items())},
                # R9: rendered T (one <|tool|> block per candidate tool).
                "mean_render_T": (round(mean_render_T, 3)
                                  if mean_render_T == mean_render_T else None),
                "render_T_hist": {str(k): v for k, v in sorted(render_hist.items())},
                "render_T_t1_share": (round(t1_share, 4)
                                      if t1_share == t1_share else None),
                "jitter_pack_tools": bool(JITTER_PACK_TOOLS),
                "jittered_rows": n_jittered,
                "max_tools": MAX_TOOLS,
                "tool_truncated_rows": n_truncated,
                "param_drop_match_missed": missed,
                "param_drop_unresolved": unresolved,
                "param_drop_other": param_other,
                # Phase I: the numeric delta report reads these five channels
                # directly. Every one of them is a "the model never saw this"
                # counter, so none may hide inside param_drop_other.
                "dropped_over_length": n_over_length,
                "multi_call_ambiguous_anchors": n_ambiguous,
                "n_recent_rows": n_recent_rows,
                "recent_truncated_rows": n_recent_trunc,
                "tool_snap_misses": int(drops.get("tool_snap_misses", 0)),
                "param_snap_misses": int(drops.get("param_snap_misses", 0)),
                "tensorize_errors": dict(errors),
                "timing": per_split_timing.get(in_name, {}),
            }
            print(f"wrote {on}.jsonl: {len(stats_rows)} groundable rows "
                  f"({len(raw) - len(rows)} ungroundable/Stratum-A/over-length "
                  f"filtered)")
            if n_rows_h1 == 0:
                print(f"  [!VACUOUS-H1] {on}: NO row carries an H1 anchor at all "
                      f"(pack_tools={PACK_TOOLS_EPH}, pool="
                      f"{len(TOOL_POOL) if TOOL_POOL is not None else 0}). tool_acc "
                      f"is undefined here, not 1.0 — check the tensorize errors above.")
            elif mean_cands < 1.5:
                print(f"  [!VACUOUS-H1] {on}: mean candidates/row={mean_cands:.2f} "
                      f"over {n_rows_h1} H1 rows "
                      f"(hist={dict(sorted(cand_hist.items()))}) -> tool_acc is an "
                      f"artifact and H1 gets ~zero gradient. Check pack_tools/pool.")
            else:
                print(f"  [tool] {on}: mean candidates/row={mean_cands:.2f} over "
                      f"{n_rows_h1} H1 rows "
                      f"(hist={dict(sorted(cand_hist.items()))}) -> H1 is a real "
                      f"{int(round(mean_cands))}-way discrimination task.")
            if n_rows_no_h1:
                print(f"  [i] {on}: {n_rows_no_h1} row(s) have no live H1 anchor "
                      f"(no candidates packed, or the GT anchor failed to snap).")
            # R9: T is the layout axis, reported beside candidate breadth.
            if n_render_rows:
                print(f"  [render] {on}: mean render_T={mean_render_T:.2f} over "
                      f"{n_render_rows} row(s) "
                      f"(hist={dict(sorted(render_hist.items()))}, "
                      f"T=1 share={t1_share:.1%}, jittered_rows={n_jittered})")
                if t1_share == 0.0 and JITTER_PACK_TOOLS:
                    print(f"  [!] {on}: T=1 never occurred although jitter is ON — "
                          f"every row would still be out-of-distribution for the "
                          f"oracle-tool eval layout. Check jitter_pack_tools wiring.")
                elif t1_share == 0.0:
                    print(f"  [i] {on}: T=1 never occurred and jitter is OFF "
                          f"(JITTER_PACK_TOOLS=False) -> the oracle-tool eval prompt "
                          f"(T=1) is OUT of the training distribution (R1c).")
            if n_truncated:
                print(f"  [!] {on}: tool_truncated_rows={n_truncated} (max_tools="
                      f"{MAX_TOOLS}) — native tools were DROPPED from the prompt, so "
                      f"H1 can never pick them. Raise data.max_tools or unset it.")
        if missed:
            print(f"  [!] MATCH_MISSED={missed} on {in_name}: extractable targets the matcher "
                  f"failed on -> these params get NO pointer supervision (real bug).")
        # ``render_T=<T>`` histogram keys and the capacity channel counters are
        # NOT drop reasons: they are already reported above, and printing them as
        # "[note] render_T=3=412" would bury the real audit lines.
        other = {k: v for k, v in drops.items()
                 if k not in ("match_missed", "unresolved_target")
                 and k not in _CHANNEL
                 and not k.startswith("render_T=")}
        for k in sorted(other):
            label = "BUG" if k in ("unaligned_target", "null_slot_missing") else "note"
            print(f"  [{label}] {k}={other[k]} on {in_name}")
        # Phase I: capacity channels printed explicitly. Each is a row/param the
        # model never got to learn from, which is the only reason this section
        # exists — silence here would read as "nothing was lost".
        if n_over_length:
            print(f"  [!] dropped_over_length={n_over_length} on {in_name}: rows "
                  f"longer than max_seq_len, dropped whole. If this is a large "
                  f"share, raise data.max_seq_len — those samples are unseen.")
        if n_ambiguous:
            print(f"  [i] multi_call_ambiguous_anchors={n_ambiguous} on {in_name}: "
                  f"anchors two calls share, so the pointer loss treats BOTH "
                  f"demands as valid (R0.4 honest-failure count, not a bug).")
        if n_recent_trunc:
            print(f"  [i] recent_truncated_rows={n_recent_trunc} on {in_name}: rows "
                  f"whose recent block exceeded the token budget and was trimmed.")
        if drops.get("tool_snap_misses"):
            print(f"  [!] tool_snap_miss={drops['tool_snap_misses']} on {in_name}: "
                  f"<|tool|> bridge lines the tokenizer could not map to tokens. Those "
                  f"candidates are PAD_INDEX(-1), i.e. invisible to H1.")
        if drops.get("param_snap_misses"):
            print(f"  [!] param_snap_miss={drops['param_snap_misses']} on {in_name}: "
                  f"param anchors that failed to snap; those parameters lose all "
                  f"pointer supervision for the row.")
        if unresolved:
            print(f"  [i] unresolved_target={unresolved} on {in_name}: needs "
                  f"generation/transform; out of scope for the retrieval-first fast "
                  f"path (correct exclusion).")
        for cls_name in sorted(errors):
            print(f"  [!!] tensorize raised {cls_name} x{errors[cls_name]} on {in_name} "
                  f"-> samples silently lost from training. Investigate.")
finally:
    if _shared_ex is not None:
        _shared_ex.shutdown(wait=True)
        print("[tensorize] shared fork pool closed")

print("\nLean tensorize summary:", json.dumps(stats, indent=2))
print("\nTensorize timing (per input split):", json.dumps(per_split_timing, indent=2))
print("\nParam supervision audit (Stratum-F params dropped from pointer loss):",
      json.dumps(drop_summary, indent=2))
if error_summary:
    print("\nTensorize exceptions (samples lost from training):",
          json.dumps(error_summary, indent=2))
missed_total = sum(d.get("match_missed", 0) for d in drop_summary.values())
print(f"match_missed total = {missed_total} (must be ~0; >0 means the span matcher "
      f"is losing groundable targets).")
bug_drops = sum(
    d.get("unaligned_target", 0) + d.get("null_slot_missing", 0)
    for d in drop_summary.values()
)
print(f"aligner/null-slot defects total = {bug_drops} (must be ~0; >0 means target "
      f"rows are unusable or the <|null|> candidate is missing).")
error_total = sum(v for e in error_summary.values() for v in e.values())
print(f"tensorize exceptions total = {error_total} (must be 0; >0 means samples "
      f"silently vanished from the training set).")

fingerprint = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "backbone": BACKBONE,
    "max_seq_len": MAX_SEQ_LEN,
    "max_span_width": MAX_SPAN_WIDTH,
    "max_pos": MAX_POS,
    "data_fraction": DATA_FRACTION,
    "split_inputs": SPLIT_INPUTS,
    "sample_limits": SAMPLE_LIMITS,
    "lean_flow": True,
}
fp_path = os.path.join(TENSOR_DIR, "config_fingerprint.json")
with open(fp_path, "w") as fh:
    json.dump(fingerprint, fh, indent=2, sort_keys=True)
print("wrote config fingerprint ->", fp_path)


[tool-pool] built once from hermes_train.jsonl: 2221 distinct schemas (pack_tools=3)
[tool-pool] 47326 duplicate schema name(s) collapsed; 1170 of them had a DIFFERING definition (first kept): ['json_output', 'get_movie_details', 'convert_currency', 'calculate_loan_payment', 'send_email', 'calculate_tip', 'convert_temperature', 'create_calendar_event', 'calculate_bmi', 'ExpertQAExtractor']
[tensorize] one fork pool for all splits: 12 worker(s) (cpu_count=12)
[tensorize] hermes_train.jsonl: 3920 rows in 20.6s (190.4 rows/s, 12 worker(s), read 0.12s)
wrote t2.jsonl: 3920 groundable rows (894 ungroundable/Stratum-A/over-length filtered)
  [tool] t2: mean candidates/row=2.48 over 3920 H1 rows (hist={1: 184, 2: 1938, 3: 1590, 4: 163, 5: 45}) -> H1 is a real 2-way discrimination task.
  [render] t2: mean render_T=2.48 over 3920 row(s) (hist={1: 184, 2: 1938, 3: 1590, 4: 163, 5: 45}, T=1 share=4.7%, jittered_rows=4533)
  [!] MATCH_MISSED=1225 on hermes_train.jsonl: extractable targets the mat

### Sanity: verify 0-indexed positive targets on `dev.jsonl`

Also re-runs the drop-reason audit over the *serialized* rows (`_param_drop_reasons`) to confirm the per-row classification survived serialization, and cross-checks that every param with zero positives has a recorded reason (no silent drops).

In [17]:
import json, os
from collections import Counter
dev_path = os.path.join(TENSOR_DIR, "dev.jsonl")
if os.path.exists(dev_path):
    with open(dev_path) as fh:
        rows = [json.loads(l) for l in fh if l.strip()]
    if not rows:
        print("dev.jsonl found but is empty (0 rows)")
    else:
        # In Lean format, positive candidate indices are 0-indexed (pi >= 0; -1 is padding)
        n_hit = sum(
            1 for r in rows
            if any(pi >= 0 for p_pos in r.get("pos_indices", []) for pi in p_pos)
        )
        print(f"dev.jsonl: {len(rows)} rows; rows with ≥1 positive target (pi >= 0): {n_hit} ({n_hit/len(rows):.1%})")
        r = rows[0]
        print("sample row keys:", sorted(r.keys()))
        c_len = len(r["cand_starts"][0]) if r.get("cand_starts") and len(r["cand_starts"]) > 0 else 0
        print("input_ids:", len(r.get("input_ids", [])), "seq_len; params K =", len(r.get("param_positions", [])),
              "cand (K,C) =", len(r.get("cand_starts", [])), "x", c_len)

        # ---- H1 trainability probe: T = number of <|tool|> candidate anchors
        # per row. T == 1 => softmax over a single class => CE == 0 (zero
        # gradient, H1 never trains) and argmax(-1) == 0 == gt => tool_acc is a
        # vacuous 1.0000 for ANY weights. T > 1 => H1 is a real task. This is
        # THE number that explains a "perfect" tool_acc, so it is printed on
        # its face instead of hidden behind the accuracy.
        def _tool_positions(r):
            """Unwrap tool_positions; the row may have been serialized as [[...]]."""
            tp = r.get("tool_positions") or []
            if tp and isinstance(tp[0], list):
                return tp[0]
            return tp

        tool_rows = [r for r in rows if "tool_positions" in r]
        t_hist = Counter()
        n_dead_slots = 0
        for r in tool_rows:
            # LIVE anchors only: a slot that failed to snap is PAD_INDEX (-1) and
            # is masked out of the logits, so it is NOT a choice H1 can make.
            # Counting raw list length would report T=3 for a row with one live
            # anchor plus two dead slots and claim H1 is a 3-way task.
            tp = _tool_positions(r)
            live = sum(1 for p in tp if int(p) >= 0)
            n_dead_slots += len(tp) - live
            t_hist[live] += 1
        n_tool_rows = len(tool_rows)
        n_h1_rows = sum(v for k, v in t_hist.items() if k > 0)
        mean_t = (sum(k * v for k, v in t_hist.items()) / n_h1_rows) if n_h1_rows else 0.0
        n_multi = sum(v for k, v in t_hist.items() if k > 1)
        print(f"H1 live candidates/row T: mean={mean_t:.2f} over {n_h1_rows} trainable rows; "
              f"hist={dict(sorted(t_hist.items()))}; dead (unsnapped) slots={n_dead_slots}")
        if n_tool_rows:
            print(f"  rows with T>1 (real discrimination): {n_multi}/{n_h1_rows} "
                  f"({(n_multi/max(n_h1_rows, 1)):.1%})")
        else:
            print("  no tool-supervised rows at all")
        if n_dead_slots:
            print(f"  [!] {n_dead_slots} <|tool|> anchor(s) failed to snap to tokens and "
                  f"are PAD_INDEX(-1) (invisible to H1). Non-zero means the bridge_text "
                  f"offset mapping and the tokenizer disagree.")
        if n_h1_rows and mean_t < 1.5:
            print("[!VACUOUS-H1] mean T < 1.5: tool_acc=1.0000 is an artifact — the H1 "
                  "softmax has a single class, CE is identically 0, and H1 receives NO "
                  "gradient. Set data.pack_tools > 1 in configs/default.yaml and rebuild "
                  "the tool pool (ToolPool.from_row_dicts).")
        if tool_rows:
            # tool_targets must be a valid index into [0, T) AND must point at a
            # LIVE anchor; -1 is the collate padding sentinel and must NOT appear
            # pre-collate. A target pointing at a dead slot is unbounded H1
            # supervision (the logit is -1e4, so CE explodes instead of training).
            bad_t = 0
            bad_examples = []
            gt_hist = Counter()
            for r in tool_rows:
                gt = r.get("tool_targets")
                if isinstance(gt, list):
                    gt = gt[0] if gt else -1
                tp = _tool_positions(r)
                gi = int(gt) if gt is not None else -1
                ok = 0 <= gi < len(tp) and int(tp[gi]) >= 0
                if not ok:
                    bad_t += 1
                    if len(bad_examples) < 5:
                        bad_examples.append((r.get("sample_id", "?"), gi, len(tp)))
                gt_hist[gi] += 1
            print(f"  tool_targets in-range AND live: {n_tool_rows - bad_t}/{n_tool_rows} "
                  f"(must equal rows; out-of-range/dead targets are silently wrong H1 "
                  f"supervision)")
            if bad_t:
                print(f"  [!] bad tool_targets examples (sample_id, gt, T): {bad_examples}")
            print(f"  gt tool index distribution (should be ~uniform if shuffled): "
                  f"{dict(sorted(gt_hist.items()))}")

        # ---- L1 audit on serialized rows: every supervised-clean param must be
        # explained. A param with all -1 positives and NO drop reason would be a
        # silent loss-hole. Reasons are split into EXPECTED (repairable only by
        # a slower path) vs BUG (must be ~0):
        #   expected : unresolved_target (needs generation), absent_from_gt
        #   bugs     : match_missed, unaligned_target, null_slot_missing
        reason_counts = Counter()
        unexplained = 0
        n_audited_rows = 0
        for r in rows:
            if "_param_drop_reasons" not in r:
                continue           # not a Stratum-F row: no audit applies
            n_audited_rows += 1
            reasons = r.get("_param_drop_reasons") or []
            reason_counts.update(reasons)
            n_zero_pos = sum(1 for p_pos in r.get("pos_indices", []) if not any(pi >= 0 for pi in p_pos))
            if n_zero_pos > len(reasons):
                unexplained += n_zero_pos - len(reasons)
        print(f"param drop reasons (dev, {n_audited_rows} audited Stratum-F rows):",
              dict(reason_counts) or "{}")
        print(f"params with zero positives and no recorded reason: {unexplained} "
              f"(must be 0 — otherwise loss silently ignores a groundable param)")
        EXPECTED_DROPS = ("unresolved_target", "absent_from_gt")
        BUG_DROPS = ("match_missed", "unaligned_target", "null_slot_missing")
        n_expected = sum(reason_counts.get(k, 0) for k in EXPECTED_DROPS)
        print(f"  expected exclusions (fast path out of scope): {n_expected} "
              f"{ {k: reason_counts[k] for k in EXPECTED_DROPS if reason_counts.get(k)} }")
        n_bugs = sum(reason_counts.get(k, 0) for k in BUG_DROPS)
        print(f"  supervision defects (must be 0): {n_bugs} "
              f"{ {k: reason_counts[k] for k in BUG_DROPS if reason_counts.get(k)} }")
        if reason_counts.get("match_missed"):
            print(f"[!] match_missed={reason_counts['match_missed']} on dev — matcher is losing "
                  f"extractable targets; check target_spans/token offsets.")
        if reason_counts.get("null_slot_missing"):
            print(f"[!] null_slot_missing={reason_counts['null_slot_missing']} on dev — the mask "
                  f"builder emitted no <|null|> candidate for a null GT param.")
        if reason_counts.get("unaligned_target"):
            print(f"[!] unaligned_target={reason_counts['unaligned_target']} on dev — target rows "
                  f"carry neither canonical_value nor char_start.")
else:
    print("dev.jsonl not found — run the tensorizer cell first")

dev.jsonl: 102 rows; rows with ≥1 positive target (pi >= 0): 102 (100.0%)
sample row keys: ['_lean_converted', '_n_tool_candidates', 'attention_mask', 'call_param_index', 'call_tool_targets', 'cand_ends', 'cand_starts', 'input_ids', 'n_calls', 'param_positions', 'pos_indices', 'tool_positions', 'tool_targets', 'utt_ends', 'utt_starts']
input_ids: 392 seq_len; params K = 2 cand (K,C) = 2 x 512
H1 live candidates/row T: mean=2.36 over 102 trainable rows; hist={2: 65, 3: 37}; dead (unsnapped) slots=0
  rows with T>1 (real discrimination): 102/102 (100.0%)
  tool_targets in-range AND live: 102/102 (must equal rows; out-of-range/dead targets are silently wrong H1 supervision)
  gt tool index distribution (should be ~uniform if shuffled): {0: 45, 1: 41, 2: 16}
param drop reasons (dev, 5 audited Stratum-F rows): {'match_missed': 5}
params with zero positives and no recorded reason: 0 (must be 0 — otherwise loss silently ignores a groundable param)
  expected exclusions (fast path out of scope

---
## 4. Training — Lean Flow (`scripts/train_lean.py`)

Runs [`scripts/train_lean.py`](../scripts/train_lean.py) on stage `T2` with `LeanSpanCallModel` and `LeanLossContainer` ($L = \lambda_{\text{ptr}} L_{\text{ptr}} + \lambda_{\text{tool}} L_{\text{tool}}$).
- Uses the same optimizer hyperparameters (`num_epochs=5`, `batch_size=8`, `gradient_accumulation_steps=4`, `bf16`, `CosineWarmupSchedule`) as the legacy notebook for a clean apples-to-apples comparison.

In [18]:
import yaml, os

TRAIN_FRACTION = 1.0

with open("configs/default.yaml") as fh:
    cfg = yaml.safe_load(fh)

cfg["experiment"]["name"] = EXPERIMENT_NAME
cfg["experiment"]["seed"] = SEED
cfg["experiment"]["output_dir"] = RESULTS_DIR
cfg["experiment"]["data_dir"] = TENSOR_DIR

NUM_EPOCHS = 5
cfg["optimization"]["num_epochs"] = NUM_EPOCHS
cfg["optimization"]["batch_size"] = 8
cfg["optimization"]["mixed_precision"] = "bf16"
cfg["training"]["gradient_accumulation_steps"] = 4
cfg["training"]["gradient_checkpointing"] = True
cfg["training"]["eval_splits"] = ["dev"]
cfg["training"]["eval_limit"] = 500
cfg["model"]["freeze_encoder_stages"] = []
cfg["model"]["max_seq_len"] = MAX_SEQ_LEN

# fraction is a TRAINER-side subsample; pack_tools is a TENSORIZER-side knob, so
# it must be pinned to the value the tensorize cell ACTUALLY used or the recorded
# config would disagree with the data on disk. That effective value is
# PACK_TOOLS_EPH (1 when no pool could be built), not the raw config request.
_EFFECTIVE_PACK_TOOLS = int(globals().get("PACK_TOOLS_EPH", PACK_TOOLS) or 1)
cfg.setdefault("data", {})["fraction"] = TRAIN_FRACTION
cfg["data"]["pack_tools"] = _EFFECTIVE_PACK_TOOLS

out = "configs/colab_lean.yaml"
with open(out, "w") as fh:
    yaml.safe_dump(cfg, fh)
print(f"wrote {out} (epochs={NUM_EPOCHS}, bs=8, accum=4, bf16=True, freeze=[], "
      f"data_fraction={TRAIN_FRACTION}, pack_tools={_EFFECTIVE_PACK_TOOLS})")
if _EFFECTIVE_PACK_TOOLS < 2:
    print("[!VACUOUS-H1] effective pack_tools < 2: H1 is a single-class softmax "
          "(CE == 0). Any tool_acc printed as 1.000 is an artifact.")
CONFIG_PATH = out


wrote configs/colab_lean.yaml (epochs=5, bs=8, accum=4, bf16=True, freeze=[], data_fraction=1.0, pack_tools=3)


In [19]:
# Direct in-notebook Lean training execution
from scripts.train_lean import main as train_lean_main

train_lean_main([
    "--config", CONFIG_PATH,
    "--experiment_name", EXPERIMENT_NAME,
    "--output_dir", RESULTS_DIR,
    "--data_dir", TENSOR_DIR,
    "--stages", "T2",
    "--device", DEVICE,
    "--seed", str(SEED),
])

[SpanCall Lean Train] config: configs/colab_lean.yaml
[SpanCall Lean Train] backbone=Qwen/Qwen3-Embedding-0.6B


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


[lean-train:T2] start: rows=3920 epochs=5 steps/epoch=490 total_steps=2450 bs=8


lean-train[T2]:   0%|          | 0/2450 [00:00<?, ?it/s]

[lean-train:T2] epoch 1/5 done: mean_loss=2.1385
[lean-eval:dev] span_acc=0.8378 (n=185) loss=1.2893 tool_acc=0.5294 (n=102) candidates/row=2.36
[lean-train:T2:epoch 1] NEW BEST on dev: exact_call=0.3824 -> saved to /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint
[lean-train:T2] epoch 2/5 done: mean_loss=0.6120
[lean-eval:dev] span_acc=0.8973 (n=185) loss=1.3541 tool_acc=0.6078 (n=102) candidates/row=2.36
[lean-train:T2:epoch 2] NEW BEST on dev: exact_call=0.5098 -> saved to /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint
[lean-train:T2] epoch 3/5 done: mean_loss=0.3118
[lean-eval:dev] span_acc=0.9081 (n=185) loss=1.3605 tool_acc=0.6569 (n=102) candidates/row=2.36
[lean-train:T2:epoch 3] NEW BEST on dev: exact_call=0.5784 -> saved to /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint
[lean-train:T2] epoch 4/5 done: mean_loss=0.1543
[lean-eval:dev] span_acc=0.9081 (n=185) loss=1.5804 tool_acc=0.6667 (n=102) candi

0

### Training Telemetry (Epoch Curves & Lean Loss Components)

Replays `results_lean/{exp}_seed{seed}/train_log.jsonl`:
- Per-stage summary (rows, epochs, final loss, steps)
- Per-epoch held-out eval curve (`span_acc`, `exact_call`, plus the H1 columns
  `tool_acc` / `n_tool` / `cands/row`)
- Per-epoch mean train loss
- LR + loss-component trajectory (`pointer`, `tool`)

`exact_call` (formerly `exact_payload_match`) is **row-level**: a row counts only
if every supervised param AND the selected tool are correct. `span_acc` is
per-(row, param) and is therefore not bounded by `exact_call`. A `tool_acc` of
`1.000` with `cands/row = 1.0` is a **single-class softmax artifact** (CE == 0,
H1 receives no gradient), flagged as `VACUOUS H1` below.

In [20]:
# ---- training telemetry: replay train_log.jsonl into readable tables ----
import json, os, math
from collections import defaultdict

def _num(rec, key):
    """Return a float, or None for missing / None   / NaN.

    NaN is reported as an em-dash, never formatted as a real number: a NaN means
    "never measured" and must not read as 0.000 (or pass a `< 1.5` threshold).
    """
    v = rec.get(key)
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return f if math.isfinite(f) else None

def _load_train_log():
    p = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}", "train_log.jsonl")
    if not os.path.exists(p):
        print(f"train_log.jsonl not found at {p}")
        return []
    with open(p) as fh:
        return [json.loads(l) for l in fh if l.strip()]

train_log = _load_train_log()
if train_log:
    stages = defaultdict(list)
    for rec in train_log:
        stages[rec.get("stage", "?")].append(rec)

    print("=" * 78)
    print("LEAN TRAIN LOG — STAGE SUMMARIES")
    print("=" * 78)
    for st, recs in stages.items():
        starts = [r for r in recs if r.get("event") == "stage_start"]
        ends = [r for r in recs if r.get("event") == "stage_end"]
        if ends:
            e = ends[-1]
            s = starts[-1] if starts else {}
            print(f"  {st:<6} rows={s.get('rows','?'):<7} epochs={s.get('num_epochs','?'):<4} "
                  f"final_loss={e.get('loss', 0):.4f}  steps={e.get('steps','?')}")

    print("\n" + "=" * 78)
    print("PER-EPOCH HELD-OUT EVAL CURVE (epoch_eval events)")
    print("=" * 78)
    evals = [r for r in train_log if r.get("event") == "epoch_eval"]
    if evals:
        # H1 columns (tool_acc / n_tool / cands) are printed alongside the
        # pointer metrics so a vacuous single-candidate 1.0000 is immediately
        # readable: tool_acc only means something when cands/row > 1. Missing
        # and NaN values render as "—" rather than as a plausible 0.000.
        print(f"  {'stage':<7}{'ep':>3}{'n':>5}{'span_acc':>10}{'exact_call':>12}"
              f"{'tool_acc':>10}{'n_tool':>8}{'cands/row':>11}")
        for r in evals:
            span = _num(r, "span_acc")
            exact = _num(r, "exact_call")
            tool_acc = _num(r, "tool_acc")
            n_tool = _num(r, "n_tool")
            cands = _num(r, "mean_n_candidates")
            sp = f"{span:>10.3f}" if span is not None else f"{'—':>10}"
            ex = f"{exact:>12.3f}" if exact is not None else f"{'—':>12}"
            ta = f"{tool_acc:>10.3f}" if tool_acc is not None else f"{'—':>10}"
            nt = f"{int(n_tool):>8}" if n_tool is not None else f"{'—':>8}"
            cd = f"{cands:>11.2f}" if cands is not None else f"{'—':>11}"
            if cands is None:
                vac = "  <-- H1 NOT EVALUATED"
            elif cands < 1.5:
                vac = "  <-- VACUOUS H1"
            else:
                vac = ""
            print(f"  {str(r.get('stage','?')):<7}{r.get('epoch','?'):>3}{r.get('n','?'):>5}"
                  f"{sp}{ex}{ta}{nt}{cd}{vac}")
        _cand = [_num(r, "mean_n_candidates") for r in evals]
        _cand = [c for c in _cand if c is not None]
        if _cand and max(_cand) < 1.5:
            print("\n  [!] H1 WARNING: every eval epoch had < 1.5 candidates/row, so the")
            print("      tool_acc column above is a single-class softmax artifact (CE == 0,")
            print("      H1 gets no gradient). Set data.pack_tools > 1 and rebuild the pool.")
        elif not _cand:
            print("\n  [!] H1 WARNING: no epoch recorded candidates/row, so tool_acc is not")
            print("      interpretable. Re-generate the notebook so epoch_eval emits it.")
    else:
        print("  no epoch_eval events")

    print("\n" + "=" * 78)
    print("PER-EPOCH MEAN TRAIN LOSS (epoch_end events)")
    print("=" * 78)
    ends = [r for r in train_log if r.get("event") == "epoch_end"]
    if ends:
        print(f"  {'stage':<8}{'epoch':>6}{'mean_loss':>12}")
        for r in ends:
            print(f"  {str(r.get('stage','?')):<8}{r.get('epoch','?'):>6}{r.get('mean_loss', 0):>12.4f}")
    else:
        print("  no epoch_end events")

    print("\n" + "=" * 78)
    print("LEAN LOSS-COMPONENT TRAJECTORY (train events, downsampled to ~40 rows)")
    print("=" * 78)
    trs = [r for r in train_log if r.get("event") == "train"]
    if trs:
        step = max(1, len(trs) // 40)
        print(f"  {'stage':<8}{'step':>6}{'loss':>10}{'pointer':>10}{'tool':>10}{'lr':>12}")
        for i, r in enumerate(trs):
            if i % step == 0 or i == len(trs) - 1:
                print(f"  {str(r.get('stage','?')):<8}{r.get('stage_step','?'):>6}{r.get('loss', 0):>10.4f}"
                      f"{r.get('pointer', 0):>10.4f}{r.get('tool', 0):>10.4f}{r.get('lr', 0):>12.2e}")
    else:
        print("  no train events")
else:
    print("No training log found — run the training cell first.")


LEAN TRAIN LOG — STAGE SUMMARIES
  T2     rows=3920    epochs=?    final_loss=0.1097  steps=2450.0

PER-EPOCH HELD-OUT EVAL CURVE (epoch_eval events)
  stage   ep    n  span_acc  exact_call  tool_acc  n_tool  cands/row
  T2       1185.0     0.838       0.382     0.529     102       2.36
  T2       2185.0     0.897       0.510     0.608     102       2.36
  T2       3185.0     0.908       0.578     0.657     102       2.36
  T2       4185.0     0.908       0.578     0.667     102       2.36
  T2       5185.0     0.903       0.569     0.667     102       2.36

PER-EPOCH MEAN TRAIN LOSS (epoch_end events)
  stage    epoch   mean_loss
  T2           1      2.1385
  T2           2      0.6120
  T2           3      0.3118
  T2           4      0.1543
  T2           5      0.1097

LEAN LOSS-COMPONENT TRAJECTORY (train events, downsampled to ~40 rows)
  stage     step      loss   pointer      tool          lr
  T2           ?    7.3082    6.3174    0.9908    5.83e-06
  T2           ?    2.3588

### Pick the Best Lean Checkpoint

Selects `results_lean/{exp}_seed{seed}/best_checkpoint` (or the latest stage checkpoint).

In [21]:
import glob, os, json

exp_root = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_seed{SEED}")

def _latest_stage_cp():
    best_cp = os.path.join(exp_root, "best_checkpoint")
    if os.path.exists(os.path.join(best_cp, "model.pt")):
        return best_cp, "best_curve"
    for stage in ["T2", "t2"]:
        cps = sorted(glob.glob(os.path.join(exp_root, stage, "checkpoints", "**", "model.pt"), recursive=True))
        if cps:
            return os.path.dirname(cps[-1]), stage
    return None, None

ALLOW_STALE_CHECKPOINT = False
fp_path = os.path.join(TENSOR_DIR, "config_fingerprint.json")
fingerprint = {
    "experiment_name": EXPERIMENT_NAME,
    "seed": SEED,
    "backbone": BACKBONE,
    "max_seq_len": MAX_SEQ_LEN,
    "max_span_width": MAX_SPAN_WIDTH,
    "max_pos": MAX_POS,
    "data_fraction": DATA_FRACTION,
    "split_inputs": SPLIT_INPUTS,
    "sample_limits": SAMPLE_LIMITS,
    "lean_flow": True,
}
config_ok = True
if os.path.exists(fp_path):
    with open(fp_path) as fh:
        saved = json.load(fh)
    config_ok = (saved == fingerprint)
    if not config_ok:
        print("WARNING: TENSOR_DIR fingerprint does NOT match current config.")
else:
    print("WARNING: no config_fingerprint.json in TENSOR_DIR (run the tensorize cell first).")
    config_ok = False

CHECKPOINT_DIR = None
cp_dir, cp_stage = _latest_stage_cp()
model_init = os.path.join(RESULTS_DIR, "model_init")

if cp_dir is not None and (config_ok or ALLOW_STALE_CHECKPOINT):
    CHECKPOINT_DIR = cp_dir
    print(f"picked stage-{cp_stage} checkpoint @ {CHECKPOINT_DIR}")
elif os.path.isdir(model_init) and config_ok:
    CHECKPOINT_DIR = model_init
    print("no staged checkpoint; using model_init @", CHECKPOINT_DIR)
else:
    all_m = sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "model.pt"), recursive=True))
    if all_m and (config_ok or ALLOW_STALE_CHECKPOINT):
        CHECKPOINT_DIR = os.path.dirname(all_m[-1])
        print("fallback checkpoint @", CHECKPOINT_DIR)
    else:
        print("no usable checkpoint found")

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)
if CHECKPOINT_DIR:
    print("  contains:", sorted(os.listdir(CHECKPOINT_DIR)))


picked stage-best_curve checkpoint @ /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint
CHECKPOINT_DIR = /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint
  contains: ['model.pt', 'provenance.json']


---
## 5. Inference & Evaluation (`LeanResolver`)

Loads `LeanSpanCallModel` + `LeanResolver` and evaluates across the same 3 decoupled parts as the legacy notebook:
1. **Eval Part 1 — Parameter Extraction Evaluation (Oracle / Given GT Tool)**: `resolver.try_resolve(gt_tool, ...)`
2. **Eval Part 2 — Function (Tool) Prediction Evaluation**: `resolver.predict_tool(candidates, ...)`
3. **Eval Part 3 — End-to-End Function Calling Evaluation**: `resolver.try_resolve_candidates(s.tools, ...)` vs `B1a` (and vs Legacy SpanCall if present).


In [22]:
import json, os
from typing import Any, Dict, List, Optional

import torch
from transformers import AutoTokenizer

from span_template import SPECIAL_MARKER_TOKENS
from spancall.data.schema import UnifiedSample
from spancall.lean.model import LeanSpanCallModel
from spancall.lean.resolver import LeanResolver
from spancall.model.encoder import SpanCallTransformerEncoder
from spancall.train.config import TrainingConfig

cfg = TrainingConfig.from_yaml(CONFIG_PATH)
tokenizer = AutoTokenizer.from_pretrained(cfg.model.backbone_name, use_fast=True)
encoder = SpanCallTransformerEncoder(
    model_name_or_path=cfg.model.backbone_name,
    hidden_dropout_prob=cfg.model.dropout,
    tokenizer=tokenizer,
)
encoder.add_marker_tokens(list(SPECIAL_MARKER_TOKENS))
model = LeanSpanCallModel(
    encoder=encoder,
    head_dim=cfg.model.head_dim,
    max_span_width=cfg.model.max_span_width,
    hidden_dim=encoder.hidden_dim,
)

# R0.4: the call budget MUST come from config. A resolver left on the default
# ``max_calls=1`` can never report more than one predicted call, so every
# multi-call metric computed from its predictions (``pred_n_calls_hist``,
# ``call_exact_set_match``) is pinned to the single-call share by construction
# — a structural cap that looks like model failure in the run log.
MAX_CALLS = int(getattr(cfg.data, "max_calls", 1))
CALL_MIN_PROB = float(getattr(cfg.data, "call_min_prob", 0.25))
CALL_MIN_PROB_RATIO = float(getattr(cfg.data, "call_min_prob_ratio", 0.5))

resolver = LeanResolver(
    model_path=CHECKPOINT_DIR,
    encoder=encoder,
    model=model,
    tokenizer=tokenizer,
    max_seq_len=cfg.model.max_seq_len,
    max_span_width=cfg.model.max_span_width,
    max_calls=MAX_CALLS,
    call_min_prob=CALL_MIN_PROB,
    call_min_prob_ratio=CALL_MIN_PROB_RATIO,
)
print(f"resolver call budget: max_calls={MAX_CALLS} "
      f"call_min_prob={CALL_MIN_PROB} call_min_prob_ratio={CALL_MIN_PROB_RATIO}")
if MAX_CALLS <= 1:
    print("  [!] max_calls=1: multi-call predictions are structurally impossible;")
    print("      call_exact_set_match / pred_n_calls_hist will be capped at the")
    print("      single-call share. Set data.max_calls in the YAML to measure them.")
if DEVICE in ("cuda", "mps"):
    model.to(DEVICE)
model.eval()
if encoder is not None and hasattr(encoder, "eval"):
    encoder.eval()
print("LeanResolver ready (checkpoint =", CHECKPOINT_DIR, ")")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

resolver call budget: max_calls=4 call_min_prob=0.25 call_min_prob_ratio=0.5
LeanResolver ready (checkpoint = /content/spancall/results_lean/spancall_lean_colab_seed42/best_checkpoint )


---
## 5a. Evaluation Part 1 — Lean Parameter Extraction (Oracle Tool Given, Retrieval-First)

Evaluates `resolver.try_resolve(gt_tool, ...)` directly on ground-truth tools. Because Lean Flow has no abstain head or gate vetoes, `res.payload_partial` and `res.parameters[p].value` directly reflect the decoded pointer predictions (with R4 equal-value aggregation and greedy top-1 fallback).

**Retrieval-first contract** (`docs/param_extraction_eval_coverage.md`): the pointer head can only *retrieve* a value that already exists in the context (utterance, recent context, time, enum, boolean, null). Params whose ground truth is `unresolved` (needs generation/transformation) are **out of scope** — refusing them is correct behaviour and routes the call to a slower path. So this cell reports:

- **extraction_recall / precision / F1** — headline quality, scored over *extractable* params only (the fast path's actual job). `n/a` means the denominator was empty (no predictions / no params), **not** a perfect score.
- **fast_path_param_coverage / fast_path_call_coverage** — how much of the workload the fast path can serve at all.
- **refused_unextractable vs missed_extractable** — splits the old `pointer_empty` bucket into *correct refusals* (routing signal) and *real fast-path failures* (the number to drive down).
- **recall_all** — legacy blended view over every GT arg, kept as a diagnostic only (always ≤ extractable recall by construction).
- **pointer_wrong** — predicted a value where GT expects a different value *or* `null`. Null-param false positives are counted here (they used to be invisible).
- **exact_payload_match_F** — full-call match; **resolver_hit_rate_F** is the resolver's own schema-validity flag and is *not* GT accuracy (kept as a separate, clearly named diagnostic).
- **E3 aligner drift** — Stratum-F samples whose GT args lack an extractable target. Non-zero is a **data** bug (aligner drift), not a model miss; it is printed with up to 10 example `sample_id:param` pairs.
- **truncation** — `evaluated / total_available` so a `limit`-capped run is never mistaken for a full-split number.

Explicit `null` GT arguments are part of the GT universe (the fast path should select `<|null|>`), so they count in coverage and every denominator.

Implementation lives in `spancall/lean/eval_p1.py` so it is unit-tested, not notebook-local.

In [23]:
import math
from spancall.lean.eval_p1 import eval_param_extraction_lean

def _m(key):
    """Read a required metric; NaN renders as n/a (empty denominator, not 1.0)."""
    v = p1_metrics[key]              # KeyError is intentional: no silent fallback
    return "n/a" if isinstance(v, float) and math.isnan(v) else v

# Retrieval-first quality metrics live in spancall/lean/eval_p1.py (unit-tested).
# NOTE: raises FileNotFoundError if EVAL_SPLIT was never built — loud by design.
p1_metrics = eval_param_extraction_lean(
    EVAL_SPLIT,
    resolver=resolver,
    data_dir=DATA_DIR,
    limit=EVAL_LIMIT,
    max_param_gap_report=25,
)

tr = p1_metrics["truncation"]
print("\n--- Fast-path value proposition ---")
print(f"split                  = {p1_metrics['split']}  "
      f"(scored {tr['evaluated']} / {tr['total_available']} available"
      f"{', TRUNCATED' if tr['truncated'] else ''})")
print(f"extraction_recall        = {_m('extraction_recall')}")
print(f"extraction_precision     = {_m('extraction_precision')}")
print(f"extraction_f1            = {_m('extraction_f1')}")
print(f"fast_path_param_coverage = {_m('fast_path_param_coverage')}")
print(f"fast_path_call_coverage  = {_m('fast_path_call_coverage')}")
print(f"correct refusals         = {_m('refused_unextractable')} "
      f"| real fast-path misses = {_m('missed_extractable')} "
      f"| pointer_wrong = {_m('pointer_wrong')}")
print(f"E3 aligner drift         = {_m('e3_aligner_drift')} sample(s) "
      f"(>0 is a DATA bug, not a model miss)")

# Phase I / J6: the call axis and the prompt layout, both read straight off the
# metrics dict so a multi-call or T=1 run is legible without re-deriving it.
print(f"\n--- Call axis & layout (J6 / R0.4) ---")
print(f"call_exact_set_match     = {_m('call_exact_set_match')} "
      f"({_m('call_exact_set_match_n')} / {_m('n')} row(s), all calls exact)")
print(f"exact_payload_match_F    = {_m('exact_payload_match_F')}  <- CALL 0 ONLY; "
      f"compare against the set number above, not in place of it")
print(f"call_count_acc           = {_m('call_count_acc')} "
      f"({_m('call_count_correct_n')} / {_m('n')})")
print(f"n_calls (GT)             = {_m('n_calls_hist')}  "
      f"multi-call rows={_m('n_multi_call_samples')}")
print(f"n_calls (predicted)      = {_m('pred_n_calls_hist')}")
print(f"mean_render_T            = {_m('mean_render_T')} "
      f"hist={_m('render_T_hist')} T=1 share={_m('render_T_t1_share')}")
_rb = p1_metrics.get("recent_budget_effective") or {}
if _rb:
    print(f"recent_budget_effective  = {_rb}")
if _m("render_T_t1_share") == 1.0:
    print("  [!] every Part-1 prompt rendered T=1: the layouts here are the "
          "oracle-tool ones, so any multi-call claim must be read against "
          "Part 3 as well.")

=== Retrieval-First Param Extraction (Oracle Tool) on hermes_eval (N=500 of 3426 available) ===
  [!] truncated by limit: scored 500 of 3426 records
HEADLINE (extractable params only): Prec=0.0560 Rec=0.1273 F1=0.0778
  -> legacy blended view (all GT args): Prec=0.0514 Rec=0.0533 F1=0.0523 (<= headline by construction)
  -> generation/out-of-scope gap: recall_all - extraction_recall = -0.0740
Exact Payload Match (Stratum F): 0.0000 (0 / 113)
Resolver Schema-Valid Hit Rate (Stratum F): 0.6903 (78 / 113)
Extraction Rate (>=1 non-null param): 0.4500
Mean candidate confidence: 0.1205
Prompt layout: mean render_T=1.2523 hist={'1': 332, '2': 112} over 444 row(s); T=1 share=0.7477
Recent-context budget (effective, post-clamp): {'2048': 500}
Call count: GT hist={'1': 500} pred hist={'1': 500} | count acc=1.0000 (500 / 500)
Call set exact match: all=0.0000 (0 / 500) | Stratum F=0.0000 (multi-call samples=0)
FAST-PATH COVERAGE: 0.4430 of GT params are extractable; 0.2260 of calls fully extractab

---
## 5b. Evaluation Part 2 — Lean Function (Tool) Prediction (H1)

Evaluates `resolver.predict_tool(candidates, ...)` across ≥ 2 candidate schemas per sample.

In [24]:
import json, os
from collections import Counter, defaultdict
from spancall.data.schema import UnifiedSample

def eval_tool_prediction_lean(split_name: str, limit: Optional[int] = 500,
                              distractor_pool: Optional[dict] = None):
    src = os.path.join(DATA_DIR, f"{split_name}.jsonl")
    if not os.path.exists(src):
        print(f"File not found: {src}")
        return

    rows = []
    if distractor_pool is None:
        distractor_pool = {}
        with open(src) as fh:
            for line in fh:
                if not line.strip():
                    continue
                s = UnifiedSample.from_dict(json.loads(line))
                rows.append(s)
                for t in s.tools or []:
                    distractor_pool.setdefault(t.name, t)
    else:
        with open(src) as fh:
            for line in fh:
                if not line.strip():
                    continue
                rows.append(UnifiedSample.from_dict(json.loads(line)))
    distractor_names = sorted(distractor_pool)

    total = 0
    multi_total = 0
    multi_correct = 0
    single_total = 0
    single_correct = 0
    augmented = 0
    probs = []
    margins = []
    empty_scores = 0
    # Phase I: T is the layout axis H1 actually discriminated over. A mean of
    # 1.0 makes ``tool_acc`` a single-class softmax artifact, so it must be
    # printed beside the accuracy, never left to be inferred.
    cand_counts = Counter()
    wrong_as = Counter()
    per_tool = defaultdict(lambda: {"n": 0, "correct": 0, "wrong_as": Counter()})

    for s in rows:
        if not s.ground_truth_calls or not s.tools:
            continue
        gt_name = s.ground_truth_calls[0].tool_name
        native_multi = len(s.tools) > 1

        candidates = list(s.tools)
        names = {t.name for t in candidates}
        grew = False
        if gt_name not in names and gt_name in distractor_pool:
            candidates.append(distractor_pool[gt_name])
            names.add(gt_name)
            grew = True
        if len(names) < 2:
            distractor = next((distractor_pool[n] for n in distractor_names if n != gt_name), None)
            if distractor is None:
                continue
            candidates.append(distractor)
            grew = True
        if grew:
            augmented += 1

        pred = resolver.predict_tool(candidates, s.utterance, s.recent_context, s.current_time_iso)
        pred_name = pred.get("tool_name")
        is_correct = bool(pred_name) and pred_name == gt_name
        total += 1
        probs.append(pred.get("tool_prob", 0.0))
        margins.append(pred.get("tool_margin", 0.0))
        if not pred.get("tool_scores"):
            empty_scores += 1
        # Phase I: count how many candidate schemas H1 actually chose between.
        _scores = pred.get("tool_scores") or {}
        cand_counts[len(_scores)] += 1

        if native_multi:
            multi_total += 1
            multi_correct += int(is_correct)
        else:
            single_total += 1
            single_correct += int(is_correct)
        if pred_name and pred_name != gt_name:
            wrong_as[pred_name] += 1
        tt = per_tool[gt_name]
        tt["n"] += 1
        tt["correct"] += int(is_correct)
        if pred_name and pred_name != gt_name:
            tt["wrong_as"][pred_name] += 1

        if limit is not None and total >= limit:
            break

    acc_all = (single_correct + multi_correct) / max(1, total)
    acc_multi = multi_correct / max(1, multi_total)
    acc_single = single_correct / max(1, single_total)
    mean_p = sum(probs) / max(1, len(probs))
    mean_m = sum(margins) / max(1, len(margins))

    mean_cands = (sum(c * n for c, n in cand_counts.items()) / max(1, total))
    print(f"=== Lean Tool Prediction (H1) on {split_name} (N={total}) — >=2 candidates ===")
    print(f"Top-1 Tool Accuracy (Overall): {acc_all:.4f} ({(single_correct + multi_correct)}/{total})")
    print(f"  Multi-tool subset: {acc_multi:.4f} ({multi_correct}/{multi_total})")
    print(f"  Single+distractor subset: {acc_single:.4f} ({single_correct}/{single_total})")
    print(f"  mean_n_candidates: {mean_cands:.2f} "
          f"(hist={dict(sorted(cand_counts.items()))})")
    if total and mean_cands < 1.5:
        print("  [!VACUOUS-H1] mean_n_candidates < 1.5: H1 had ONE class per row, so "
              "the accuracy above is an artifact, not tool-selection skill.")
    print(f"Mean Winner Tool Probability: {mean_p:.4f} | Mean Margin: {mean_m:.4f}")
    print(f"Rows with empty tool_scores: {empty_scores}")
    print(f"Rows where candidates were augmented beyond s.tools: {augmented}")
    print(f"Most common wrong picks: {wrong_as.most_common(8)}")

    if per_tool:
        print("\n--- Per-tool H1 accuracy ---")
        print(f"{'tool':<30}{'n':>6}{'acc':>8}{'wrong_as':<40}")
        for tname, st in sorted(per_tool.items(), key=lambda kv: (-kv[1]["n"], kv[1]["correct"])):
            acc_t = st["correct"] / max(1, st["n"])
            wrong_str = ", ".join(f"{w}x{v}" for w, v in st["wrong_as"].most_common(3))
            print(f"{tname:<30}{st['n']:>6}{acc_t:>8.3f}  {wrong_str}")
    print("==================================================================")

eval_tool_prediction_lean(EVAL_SPLIT, limit=EVAL_LIMIT)

=== Lean Tool Prediction (H1) on hermes_eval (N=500) — >=2 candidates ===
Top-1 Tool Accuracy (Overall): 0.8100 (405/500)
  Multi-tool subset: 0.0000 (0/0)
  Single+distractor subset: 0.8100 (405/500)
  mean_n_candidates: 1.70 (hist={0: 75, 2: 425})
Mean Winner Tool Probability: 0.4849 | Mean Margin: 0.1198
Rows with empty tool_scores: 75
Rows where candidates were augmented beyond s.tools: 500
Most common wrong picks: [('ActiveDrillingProjects', 11), ('ExpertQAExtractor', 9)]

--- Per-tool H1 accuracy ---
tool                               n     accwrong_as                                
json_output                      500   0.810  ActiveDrillingProjectsx11, ExpertQAExtractorx9


In [25]:
def _param_detail(pr) -> Dict[str, Any]:
    return {
        "value": getattr(pr, "value", None),
        "confidence": float(getattr(pr, "confidence", 0.0) or 0.0),
        "margin": float(getattr(pr, "margin", 0.0) or 0.0),
        "source_type": getattr(pr, "source_type", None),
        "char_span": list(pr.char_span) if getattr(pr, "char_span", None) else None,
        "p_abstain": 0.0,
    }


def _features_detail(features) -> Dict[str, Any]:
    feats = dict(features or {})
    if "tool_scores" in feats and isinstance(feats["tool_scores"], dict):
        top = sorted(feats["tool_scores"].items(), key=lambda kv: -float(kv[1]))[:3]
        feats["tool_scores"] = {k: float(v) for k, v in top}
    return {k: v for k, v in feats.items()
            if k not in ("recent_selected", "tool_scores_all")}


def predict_split(split_name: str, out_path: str, limit: Optional[int] = None,
                  detailed: bool = True) -> None:
    src = os.path.join(DATA_DIR, f"{split_name}.jsonl")
    sub_src = os.path.join(RESULTS_DIR, f"subset_{split_name}.jsonl")
    if not os.path.exists(src):
        print(f"no source {src} — skipped")
        return
    preds: List[Dict[str, Any]] = []
    subset_samples: List[Dict[str, Any]] = []
    n = 0
    with open(src) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            raw_dict = json.loads(line)
            s = UnifiedSample.from_dict(raw_dict)
            res = resolver.try_resolve_candidates(s.tools, s.utterance, s.recent_context, s.current_time_iso)
            rec = {
                "sample_id": s.sample_id,
                "stratum": s.stratum.value,
                "gt_tool": (s.ground_truth_calls[0].tool_name if s.ground_truth_calls else None),
                "gt_args": (s.ground_truth_calls[0].arguments if s.ground_truth_calls else {}),
                # R0.4 / J6: keep the WHOLE gt call list so the evaluator can score
                # the predicted call SET instead of only the flattened first call.
                "gt_calls": [{"tool_name": c.tool_name, "arguments": dict(c.arguments or {})}
                             for c in (s.ground_truth_calls or [])],
                "n_gt_calls": len(s.ground_truth_calls or []),
                # R0.4: the decoded call list. ``tool_name``/``payload`` above stay
                # the primary (call-0) view, so single-call consumers are unchanged.
                "calls": [c.as_dict() for c in (getattr(res, "calls", None) or [])],
                "n_pred_calls": len(getattr(res, "calls", None) or []),
                "hit": bool(res.hit),
                "status": getattr(res, "status", "hit" if res.hit else "partial"),
                "payload": res.payload,
                "payload_partial": getattr(res, "payload_partial", {}),
                "missing_required": getattr(res, "missing_required", []),
                "missing_optional": getattr(res, "missing_optional", []),
                "tool_name": res.tool_name,
                "confidence": float(getattr(res, "confidence", res.accept_score) or 0.0),
                "accept_probability": float(res.accept_score or 0.0),
                "joint_margin": float(getattr(res, "joint_margin", 0.0) or 0.0),
                "abstain_reason": res.abstain_reason,
                "latency_ms": float(getattr(res, "latency_ms", 0.0) or 0.0),
            }
            if detailed:
                rec["parameters"] = {p: _param_detail(pr) for p, pr in (res.parameters or {}).items()}
                rec["features"] = _features_detail(getattr(res, "features", {}))
            preds.append(rec)
            subset_samples.append(raw_dict)
            n += 1
            if limit is not None and n >= limit:
                break
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fh:
        for p in preds:
            fh.write(json.dumps(p, default=str) + "\n")
    with open(sub_src, "w") as fh:
        for s_dict in subset_samples:
            fh.write(json.dumps(s_dict, default=str) + "\n")
    n_hit = sum(1 for p in preds if p["hit"])
    print(f"predicted {n} rows → {out_path} (hit={n_hit}) and saved subset → {sub_src}")

predict_split(EVAL_SPLIT, os.path.join(RESULTS_DIR, "preds_eval.jsonl"), limit=EVAL_LIMIT)

predicted 500 rows → /content/spancall/results_lean/preds_eval.jsonl (hit=170) and saved subset → /content/spancall/results_lean/subset_hermes_eval.jsonl


---
## 5c. Sample-Prediction Forensics

Inspects per-sample predictions in `preds_eval.jsonl`: hits, wrong-tool picks, and partial/schema-invalid payloads.

In [26]:
import json, os, textwrap
from collections import Counter

PREDS_PATH = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
if not os.path.exists(PREDS_PATH):
    print("preds_eval.jsonl not found — run the predict_split cell first")
else:
    with open(PREDS_PATH) as fh:
        preds = [json.loads(l) for l in fh if l.strip()]
    subs = {}
    sub_path = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}.jsonl")
    if os.path.exists(sub_path):
        from spancall.data.schema import UnifiedSample
        with open(sub_path) as fh:
            for l in fh:
                if l.strip():
                    d = json.loads(l)
                    subs[d.get("sample_id")] = UnifiedSample.from_dict(d)

    def _utt(sid):
        s = subs.get(sid)
        return textwrap.shorten(s.utterance, width=160, placeholder=" …") if s else "(n/a)"

    def _verdict(p):
        if p.get("tool_name") and p.get("gt_tool") and p["tool_name"] != p["gt_tool"]:
            return "WRONG_TOOL"
        if p.get("hit"):
            return "HIT"
        return "PARTIAL"

    verdicts = Counter(_verdict(p) for p in preds)
    print(f"Loaded {len(preds)} Lean prediction records from {PREDS_PATH}")
    print("Verdict distribution:")
    for v, c in verdicts.most_common():
        print(f"  {v:<12}{c:>5}  ({c / max(1, len(preds)):.1%})")

    hits = [p for p in preds if p.get("hit")]
    print("\n" + "=" * 72)
    print(f"SAMPLE HITS ({len(hits)} total; showing up to 5)")
    print("=" * 72)
    for p in hits[:5]:
        print(f"\n--- {p.get('sample_id')} [{p.get('tool_name')}] conf={p.get('confidence'):.3f} ---")
        print(f"  utt : {_utt(p.get('sample_id'))}")
        print(f"  gt  : {json.dumps(p.get('gt_args'), default=str)}")
        print(f"  pred: {json.dumps(p.get('payload'), default=str)}")

    wrong = [p for p in preds if _verdict(p) == "WRONG_TOOL"]
    if wrong:
        print("\n" + "=" * 72)
        print(f"WRONG-TOOL MISSES ({len(wrong)} total; showing up to 5)")
        print("=" * 72)
        for p in wrong[:5]:
            print(f"\n--- {p.get('sample_id')} → predicted {p.get('tool_name')} (gt={p.get('gt_tool')}) ---")
            print(f"  utt : {_utt(p.get('sample_id'))}")
            if "features" in p and "tool_scores" in (p.get("features") or {}):
                print(f"  H1 scores: {p['features']['tool_scores']}")

    part = [p for p in preds if _verdict(p) == "PARTIAL"]
    if part:
        print("\n" + "=" * 72)
        print(f"PARTIAL / SCHEMA-INVALID MISSES ({len(part)} total; showing up to 5)")
        print("=" * 72)
        for p in part[:5]:
            print(f"\n--- {p.get('sample_id')} [{p.get('tool_name')}] reason={p.get('abstain_reason')} ---")
            print(f"  utt : {_utt(p.get('sample_id'))}")
            print(f"  gt  : {json.dumps(p.get('gt_args'), default=str)}")
            print(f"  pred: {json.dumps(p.get('payload_partial'), default=str)}")

Loaded 500 Lean prediction records from /content/spancall/results_lean/preds_eval.jsonl
Verdict distribution:
  PARTIAL       280  (56.0%)
  WRONG_TOOL    122  (24.4%)
  HIT            98  (19.6%)

SAMPLE HITS (170 total; showing up to 5)

--- json_mode_agentic::fccba797-94f6-40c2-a480-64f9ba52db78 [json_output] conf=0.035 ---
  utt : I'm working on a literature review and I've come across an article that references several academic papers. I need to extract the titles and authors of these …
  gt  : {"papers": [{"title": "The Structure of Scientific Revolutions", "author": "Thomas Kuhn"}, {"title": "On the Electrodynamics of Moving Bodies", "author": null}]}
  pred: {"papers": ["I"]}

--- json_mode_agentic::f2b2eb03-9327-44d9-9583-82cea2985b84::r7 [json_output] conf=1.000 ---
  utt : I'm working on a project that involves analyzing action descriptions and extracting structured data from them. Specifically, I need to generate (subject, …
  gt  : {"Event Triple": {"subject": "Taylor", "p

---
## 6. General Function Calling Evaluation (Eval Part 3)

Runs [`scripts/eval_toolcall.py`](../scripts/eval_toolcall.py) on identical subsets (`subset_{EVAL_SPLIT}.jsonl`):
1. **B1a baseline** (`--baseline b1a`)
2. **SpanCall Lean** (`--predictions preds_eval.jsonl`)
3. *(Automatic if present)* **Legacy SpanCall** (`results/toolcall_metrics_{EVAL_SPLIT}_model.json`) for direct comparison.

In [27]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/eval_toolcall.py",
    "--test_data", f"{RESULTS_DIR}/subset_{EVAL_SPLIT}.jsonl",
    "--baseline", "b1a",
    "--output", f"{RESULTS_DIR}/toolcall_metrics_{EVAL_SPLIT}_b1a.json",
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/eval_toolcall.py', '--test_data', '/content/spancall/results_lean/subset_hermes_eval.jsonl', '--baseline', 'b1a', '--output', '/content/spancall/results_lean/toolcall_metrics_hermes_eval_b1a.json'], returncode=0)

In [28]:
import subprocess, sys
subprocess.run([
    sys.executable, "scripts/eval_toolcall.py",
    "--test_data", f"{RESULTS_DIR}/subset_{EVAL_SPLIT}.jsonl",
    "--predictions", f"{RESULTS_DIR}/preds_eval.jsonl",
    "--output", f"{RESULTS_DIR}/toolcall_metrics_{EVAL_SPLIT}_lean.json",
], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/eval_toolcall.py', '--test_data', '/content/spancall/results_lean/subset_hermes_eval.jsonl', '--predictions', '/content/spancall/results_lean/preds_eval.jsonl', '--output', '/content/spancall/results_lean/toolcall_metrics_hermes_eval_lean.json'], returncode=0)

### Headline Comparison Table (B1a vs Legacy vs Lean Flow)

In [29]:
import json, os

def _load_metrics(path):
    if not os.path.exists(path):
        return None
    with open(path) as fh:
        return json.load(fh)

rows = []
for label, path in [
    ("B1a (regex)", os.path.join(RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_b1a.json")),
    ("SpanCall (legacy)", os.path.join(LEGACY_RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_model.json")),
    ("SpanCall (LEAN)", os.path.join(RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_lean.json")),
]:
    m = _load_metrics(path)
    if m is None:
        continue
    m_f = m.get("metrics_by_stratum", {}).get("F", {})
    rows.append({
        "system": label,
        "n_F": int(m_f.get("n", 0)),
        "exact_F": round(m_f.get("exact_payload_match", 0.0), 3),
        # R0.4 / J6: set-level headline. ``exact_F`` above stays call-0-only, so
        # a two-call row can be "exact" while its SET is wrong (dropped/extra call).
        "set_exact_F": round(m_f.get("call_exact_set_match", 0.0), 3),
        "count_acc": round(m.get("call_count_acc", 0.0), 3),
        "hit_F": round(m_f.get("hit_rate", 0.0), 3),
        "param_F1_F": round(m_f.get("param_f1", 0.0), 3),
        "false_hit_on_A": round(m.get("false_hit_rate_on_a", 0.0), 4),
        "exact_all": round(m.get("exact_payload_match", 0.0), 3),
        "set_exact_all": round(m.get("call_exact_set_match", 0.0), 3),
        "hit_all": round(m.get("hit_rate", 0.0), 3),
        "n_calls": m.get("n_calls_hist", {}),
    })

if rows:
    headers = ["system", "n_F", "exact_F", "set_exact_F", "count_acc", "hit_F",
               "param_F1_F", "false_hit_on_A", "exact_all", "set_exact_all",
               "hit_all", "n_calls"]
    print(" | ".join(headers))
    print("-|" * len(headers))
    for r in rows:
        print(" | ".join(str(r[h]) for h in headers))
else:
    print("no metric files found yet — run the eval_toolcall cells above")

system | n_F | exact_F | set_exact_F | count_acc | hit_F | param_F1_F | false_hit_on_A | exact_all | set_exact_all | hit_all | n_calls
-|-|-|-|-|-|-|-|-|-|-|-|
B1a (regex) | 65 | 0.0 | 0.0 | 1.0 | 0.031 | 0.007 | 0.0092 | 0.0 | 0.0 | 0.012 | {'1': 500}
SpanCall (LEAN) | 65 | 0.0 | 0.0 | 1.0 | 0.508 | 0.014 | 0.3149 | 0.0 | 0.0 | 0.34 | {'1': 500}


### Detailed Eval Forensics (Error Taxonomy & Per-Tool E2E)

In [30]:
import json, os
from collections import Counter, defaultdict

def _err_buckets(m):
    return {k: m.get(k, 0) for k in [
        "error_wrong_tool", "error_wrong_value", "error_entity_swap",
        "error_duplicate_span", "error_date_misparse", "error_extra_call", "error_missing_call"]}

m_path = os.path.join(RESULTS_DIR, f"toolcall_metrics_{EVAL_SPLIT}_lean.json")
if not os.path.exists(m_path):
    print("Lean metrics file not found — run the eval_toolcall cell first")
else:
    with open(m_path) as fh:
        m = json.load(fh)

    print("=" * 72)
    print(f"ERROR TAXONOMY — {EVAL_SPLIT} (SpanCall Lean)")
    print("=" * 72)
    errs = _err_buckets(m)
    tot_err = sum(errs.values())
    for k, v in sorted(errs.items(), key=lambda kv: -kv[1]):
        if v:
            print(f"  {k:<22}{v:>5}  ({v / max(1, tot_err):.1%} of errors)")

    p_path = os.path.join(RESULTS_DIR, "preds_eval.jsonl")
    sub_path = os.path.join(RESULTS_DIR, f"subset_{EVAL_SPLIT}.jsonl")
    preds, gt_map = [], {}
    if os.path.exists(p_path):
        with open(p_path) as fh:
            preds = [json.loads(l) for l in fh if l.strip()]
    if os.path.exists(sub_path):
        from spancall.eval.toolcall.evaluator import payload_equal
        from spancall.data.schema import UnifiedSample
        with open(sub_path) as fh:
            for l in fh:
                if l.strip():
                    s = UnifiedSample.from_dict(json.loads(l))
                    if s.ground_truth_calls:
                        gt_map[s.sample_id] = s
        for p in preds:
            s = gt_map.get(p.get("sample_id"))
            if s is not None and s.ground_truth_calls:
                gt = s.ground_truth_calls[0]
                p["gt_tool"] = gt.tool_name
                p["payload_equal"] = payload_equal(gt.arguments or {}, p.get("payload"),
                                                   s.tools[0] if s.tools else None)

    print("\n" + "=" * 72)
    print("PER-TOOL E2E (SpanCall Lean)")
    print("=" * 72)
    per_tool = defaultdict(lambda: {"n": 0, "hit": 0, "exact": 0})
    for p in preds:
        tname = p.get("gt_tool") or p.get("tool_name")
        per_tool[tname]["n"] += 1
        per_tool[tname]["hit"] += int(bool(p.get("hit")))
        per_tool[tname]["exact"] += int(bool(p.get("payload_equal")))
    print(f"{'tool':<30}{'n':>6}{'hit':>6}{'hit%':>7}{'exact':>7}{'exact%':>8}")
    for tname, st in sorted(per_tool.items(), key=lambda kv: -kv[1]["n"]):
        if st["n"] == 0:
            continue
        print(f"{tname:<30}{st['n']:>6}{st['hit']:>6}{st['hit'] / st['n']:>7.2f}"
              f"{st['exact']:>7}{st['exact'] / st['n']:>8.2f}")

    # R9: the render count T the resolver actually built per row, straight off the
    # persisted ``features`` block. Without this the eval table cannot be read:
    # a T=1 layout makes H1 a single-class softmax whose accuracy is 1.0 for ANY
    # weights, so a high ``hit`` rate there is not evidence of tool selection.
    print("\n" + "=" * 72)
    print("PROMPT LAYOUT (render_T) & RECENT BUDGET — SpanCall Lean")
    print("=" * 72)
    rt_hist: Counter = Counter()
    rb_hist: Counter = Counter()
    n_rt = 0
    for p in preds:
        feats = p.get("features") or {}
        rt = feats.get("render_T")
        if rt is not None:
            try:
                rt_hist[int(rt)] += 1
                n_rt += 1
            except (TypeError, ValueError):
                pass
        rb = feats.get("recent_budget_effective")
        if rb is not None:
            try:
                rb_hist[int(rb)] += 1
            except (TypeError, ValueError):
                pass

    if not n_rt:
        print("  render_T not reported by this resolver build — layout UNKNOWN")
        print("  (re-run the predict cell with a resolver that sets features['render_T'])")
    else:
        mean_T = sum(t * c for t, c in rt_hist.items()) / n_rt
        t1_share = rt_hist.get(1, 0) / n_rt
        print(f"  mean render_T = {mean_T:.2f} over {n_rt} row(s)")
        print(f"  hist          = {dict(sorted(rt_hist.items()))}")
        print(f"  T=1 share     = {t1_share:.1%}")
        if t1_share == 1.0:
            print("  [!] EVERY eval prompt rendered exactly ONE tool block. H1's softmax")
            print("      is single-class there: tool_acc = 1.0 regardless of the weights,")
            print("      so the hit rates above are NOT evidence of tool-selection skill.")
        elif t1_share == 0.0:
            print("  [!] T=1 never occurred in eval although training jitters T down to 1.")
            print("      Eval is then stricter than training on the T axis (R1).")
    if rb_hist:
        print(f"  recent_budget_effective = {dict(sorted(rb_hist.items()))}")
    else:
        print("  recent_budget_effective not reported (no clamp telemetry on these rows)")

    # R0.4 / J6: the call-axis view. ``exact_payload_match`` scores ONLY call 0,
    # so on the multi-call rows it silently forgives a dropped or invented second
    # call. These lines make that visible instead of trusting the headline.
    print("\n" + "=" * 72)
    print("CALL SET (multi-call supervision, R0.4) — SpanCall Lean")
    print("=" * 72)
    gt_hist: Counter = Counter()
    pred_hist: Counter = Counter()
    n_set_rows = 0
    n_set_exact = 0
    for p in preds:
        n_gt = p.get("n_gt_calls")
        n_pred = p.get("n_pred_calls")
        if n_gt is None:
            n_gt = len(p.get("gt_calls") or [])
        if n_pred is None:
            n_pred = len(p.get("calls") or [])
        gt_hist[int(n_gt)] += 1
        pred_hist[int(n_pred)] += 1
        n_set_rows += 1
        n_set_exact += int(int(n_gt) == int(n_pred))
    multi_share = (sum(c for k, c in gt_hist.items() if k > 1) / n_set_rows) if n_set_rows else 0.0
    print(f"  GT n_calls histogram  = {dict(sorted(gt_hist.items()))}")
    print(f"  pred n_calls histogram= {dict(sorted(pred_hist.items()))}")
    print(f"  multi-call share      = {multi_share:.1%} of {n_set_rows} row(s)")
    print(f"  call_count_acc (from these rows)  = {n_set_exact / max(1, n_set_rows):.3f}")
    # R0.4: the resolver's call budget is a STRUCTURAL ceiling, not model quality.
    # With max_calls=1 no row can predict a second call, so a lower SET headline
    # is the cap talking, not a dropped call.
    max_pred_calls = max(pred_hist) if pred_hist else 0
    if multi_share > 0 and max_pred_calls <= 1:
        print("  [!] STRUCTURAL CAP: every row predicted <=1 call while "
              f"{multi_share:.1%} of rows are multi-call.")
        print("      The resolver was built with max_calls=1, so the call axis was")
        print("      never decoded. This number measures configuration, not the model.")
        print("      Set data.max_calls > 1 in the YAML and rebuild the resolver.")
    set_exact_metric = m.get("call_exact_set_match")
    if set_exact_metric is not None:
        print(f"  call_exact_set_match (evaluator)  = {set_exact_metric:.3f}")
        print(f"  exact_payload_match (call-0 only) = {m.get('exact_payload_match', 0.0):.3f}")
        if multi_share > 0 and set_exact_metric < m.get("exact_payload_match", 0.0):
            print("  [!] the SET headline is BELOW the call-0 headline: at least one")
            print("      multi-call row matched call 0 but lost a call in the set.")
    else:
        print("  call_exact_set_match not present — re-run eval_toolcall.py (R0.4)")

ERROR TAXONOMY — hermes_eval (SpanCall Lean)
  error_wrong_value        81  (45.5% of errors)
  error_wrong_tool         72  (40.4% of errors)
  error_missing_call       17  (9.6% of errors)
  error_date_misparse       8  (4.5% of errors)

PER-TOOL E2E (SpanCall Lean)
tool                               n   hit   hit%  exact  exact%
json_output                      500   170   0.34      0    0.00

PROMPT LAYOUT (render_T) & RECENT BUDGET — SpanCall Lean
  mean render_T = 1.00 over 425 row(s)
  hist          = {1: 425}
  T=1 share     = 100.0%
  [!] EVERY eval prompt rendered exactly ONE tool block. H1's softmax
      is single-class there: tool_acc = 1.0 regardless of the weights,
      so the hit rates above are NOT evidence of tool-selection skill.
  recent_budget_effective = {2048: 500}

CALL SET (multi-call supervision, R0.4) — SpanCall Lean
  GT n_calls histogram  = {1: 500}
  pred n_calls histogram= {0: 75, 1: 425}
  multi-call share      = 0.0% of 500 row(s)
  call_count_acc (fro

---
## 7. Test — OOD / Adversarial Splits

Evaluates `LeanResolver` and `B1a` on the three unseen test splits (`hermes_test_ood_schema`, `hermes_test_ood_para`, `hermes_test_adversarial`).

In [31]:
for split in TEST_SPLITS:
    predict_split(split, os.path.join(RESULTS_DIR, f"preds_{split}.jsonl"), limit=TEST_LIMIT)

predicted 200 rows → /content/spancall/results_lean/preds_hermes_test_ood_schema.jsonl (hit=112) and saved subset → /content/spancall/results_lean/subset_hermes_test_ood_schema.jsonl
predicted 200 rows → /content/spancall/results_lean/preds_hermes_test_ood_para.jsonl (hit=165) and saved subset → /content/spancall/results_lean/subset_hermes_test_ood_para.jsonl
predicted 200 rows → /content/spancall/results_lean/preds_hermes_test_adversarial.jsonl (hit=68) and saved subset → /content/spancall/results_lean/subset_hermes_test_adversarial.jsonl


In [32]:
import subprocess, sys

def _run_eval(split: str, label: str, extra: list) -> None:
    out = f"{RESULTS_DIR}/toolcall_metrics_{split}_{label}.json"
    data_file = f"{RESULTS_DIR}/subset_{split}.jsonl"
    if not os.path.exists(data_file):
        data_file = f"{DATA_DIR}/{split}.jsonl"
    cmd = [
        sys.executable, "scripts/eval_toolcall.py",
        "--test_data", data_file,
        *extra,
        "--output", out,
    ]
    subprocess.run(cmd, check=False)
    print(f"eval {split} [{label}] -> {out}")

for split in TEST_SPLITS:
    _run_eval(split, "b1a", ["--baseline", "b1a"])
    _run_eval(split, "lean", ["--predictions", f"{RESULTS_DIR}/preds_{split}.jsonl"])

print("test eval done")

eval hermes_test_ood_schema [b1a] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_ood_schema_b1a.json
eval hermes_test_ood_schema [lean] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_ood_schema_lean.json
eval hermes_test_ood_para [b1a] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_ood_para_b1a.json
eval hermes_test_ood_para [lean] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_ood_para_lean.json
eval hermes_test_adversarial [b1a] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_adversarial_b1a.json
eval hermes_test_adversarial [lean] -> /content/spancall/results_lean/toolcall_metrics_hermes_test_adversarial_lean.json
test eval done


In [33]:
import json, os

print(f"{'split':<26}{'system':<16}{'n_F':>6}{'exact_F':>9}{'hit_F':>8}{'param_F1':>9}{'exact_all':>10}")
for split in TEST_SPLITS + [EVAL_SPLIT]:
    candidates = [
        ("b1a", os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_b1a.json")),
        ("legacy (raw)", os.path.join(LEGACY_RESULTS_DIR, f"toolcall_metrics_{split}_model.json")),
        ("lean", os.path.join(RESULTS_DIR, f"toolcall_metrics_{split}_lean.json")),
    ]
    for label, path in candidates:
        m = _load_metrics(path)
        if not m:
            continue
        m_f = m.get("metrics_by_stratum", {}).get("F", {})
        n_f = int(m_f.get("n", 0))
        exact_f = m_f.get("exact_payload_match", 0.0)
        hit_f = m_f.get("hit_rate", 0.0)
        pf1_f = m_f.get("param_f1", 0.0)
        exact_all = m.get("exact_payload_match", 0.0)
        print(f"{split:<26}{label:<16}{n_f:>6}"
              f"{exact_f:>9.3f}{hit_f:>8.3f}"
              f"{pf1_f:>9.3f}{exact_all:>10.3f}")

split                     system             n_F  exact_F   hit_F param_F1 exact_all
hermes_test_ood_schema    b1a                 24    0.083   0.000    0.000     0.010
hermes_test_ood_schema    lean                24    0.083   0.542    0.000     0.010
hermes_test_ood_para      b1a                200    0.010   0.020    0.018     0.010
hermes_test_ood_para      lean               200    0.030   0.825    0.124     0.030
hermes_test_adversarial   b1a                  0    0.000   0.000    0.000     0.000
hermes_test_adversarial   lean                 0    0.000   0.000    0.000     0.005
hermes_eval               b1a                 65    0.000   0.031    0.007     0.000
hermes_eval               lean                65    0.000   0.508    0.014     0.000


---
## 8. Persist to Drive (Optional)

In [34]:
if DRIVE_MOUNT:
    import shutil, os
    os.makedirs(DRIVE_OUT, exist_ok=True)
    for name, src in [
        ("data_tensorized_lean", TENSOR_DIR),
        ("results_lean", RESULTS_DIR),
    ]:
        if os.path.isdir(src):
            dst = os.path.join(DRIVE_OUT, name)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"mirrored {src} → {dst}")
    print("\nLean artifacts persisted to", DRIVE_OUT)

---
## 9. Test Suite (Lean Flow + Repo Unit Tests)

Runs `tests/test_lean_flow.py` and the full unit test suite to confirm everything passes.

In [35]:
!python -m pytest tests/test_lean_flow.py tests -q --disable-warnings --no-header 2>&1 | tail -25

........................................................................ [ 84%]
.............                                                            [100%]
85 passed in 7.76s


In [36]:
print(f"""
== SpanCall Lean Flow pipeline complete ==

What ran:
  0. git clone/pull + editable pip install
  1. env setup (.env loaded) + spancall.lean import check
  2. HF dataset ({HF_DATASET_ID}) → {DATA_DIR}
  3. 0-indexed Lean tensorizer → {TENSOR_DIR}
  4. Lean T2 training (scripts/train_lean.py) → {RESULTS_DIR}
  5. LeanResolver eval Part 1 (retrieval-first param extraction) & Part 2 (H1 Tool Selection)
  6. Eval Part 3 (End-to-End Function Calling: B1a vs Legacy vs Lean) on {EVAL_SPLIT}
  7. OOD / Adversarial test splits: {TEST_SPLITS}
  8. Optional Drive mirror @ {DRIVE_OUT}
  9. Lean Flow + repo unit tests
""")


== SpanCall Lean Flow pipeline complete ==

What ran:
  0. git clone/pull + editable pip install
  1. env setup (.env loaded) + spancall.lean import check
  2. HF dataset (convoicon/spancall) → /content/spancall/data/processed
  3. 0-indexed Lean tensorizer → /content/spancall/data/tensorized_lean
  4. Lean T2 training (scripts/train_lean.py) → /content/spancall/results_lean
  5. LeanResolver eval Part 1 (retrieval-first param extraction) & Part 2 (H1 Tool Selection)
  6. Eval Part 3 (End-to-End Function Calling: B1a vs Legacy vs Lean) on hermes_eval
  7. OOD / Adversarial test splits: ['hermes_test_ood_schema', 'hermes_test_ood_para', 'hermes_test_adversarial']
  8. Optional Drive mirror @ /content/drive/MyDrive/spancall_lean_artifacts
  9. Lean Flow + repo unit tests

